# Model 2: Spiking LSTM with Temporal Attention

PyTorch S-LSTM model using delta-sigma spike encoding and temporal attention for five-class Bonn EEG seizure-related classification.

This GitHub-ready notebook has cleared execution outputs for lightweight version control. Run the cells sequentially after mounting or configuring the Bonn EEG dataset path used in the project.


In [ ]:
# =========================================================
# CELL 1: IMPORTS, SEED, DEVICE
# =========================================================

import os
import random
import time
import copy
import math

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

# -----------------------------
# Reproducibility
# -----------------------------
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = True

# -----------------------------
# Device
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=" * 70)
print("SETUP")
print("=" * 70)
print(f"Device: {device}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# =========================================================
# CELL 2: CONFIG
# =========================================================

# Required fair setup
chunk_size = 256
stride = 128

# Full-signal split
train_ratio = 0.70
val_ratio = 0.10
test_ratio = 0.20

# Pure input spike encoding
input_spike_threshold = 0.18   # slightly richer delta-sigma spike stream

# Training
batch_size = 64
epochs = 60
learning_rate = 5e-4
weight_decay = 5e-4
# No early stopping: train for the full number of epochs.
early_stop_patience = None
label_smoothing = 0.02

# Spike-preserving training regularization
train_time_shift_max = 8        # random left/right shift in timesteps, zero-filled
input_spike_dropout_p = 0.01    # lighter spike dropout preserves weak events
use_best_checkpoint_for_test = True

# Model
input_channels = 2             # positive delta-sigma, negative delta-sigma
conv1_channels = 32
conv2_channels = 64
feature_dim = 128

hidden1 = 192
hidden2 = 128

num_classes = 5
dropout_p = 0.12

# Spiking thresholds inside model
conv_spike_threshold = 0.50
hidden_spike_threshold = 0.12

print("=" * 70)
print("CONFIG")
print("=" * 70)
print(f"chunk_size: {chunk_size}")
print(f"stride: {stride}")
print(f"split: {train_ratio:.0%} train / {val_ratio:.0%} val / {test_ratio:.0%} test")
print(f"input_spike_threshold: {input_spike_threshold}")
print(f"batch_size: {batch_size}")
print(f"epochs: {epochs}  (fixed full run, no early stopping)")
print(f"learning_rate: {learning_rate}")
print("conv channels: not used (strict no-conv pure spiking frontend)")
print(f"hidden sizes: {hidden1} -> {hidden2}")
print(f"dropout_p: {dropout_p}")
print(f"train_time_shift_max: {train_time_shift_max}")
print(f"input_spike_dropout_p: {input_spike_dropout_p}")
print(f"use_best_checkpoint_for_test: {use_best_checkpoint_for_test}")

# Figure saving
FIG_DIR = "/kaggle/working/figures_pdf"
os.makedirs(FIG_DIR, exist_ok=True)

print(f"PDF figures will be saved to: {FIG_DIR}")


In [ ]:
# =========================================================
# CELL 3: LOAD BONN EEG FULL SIGNALS
# =========================================================

DATA_ROOT = "/kaggle/input/datasets/quands/eeg-dataset/Dataset"

def find_bonn_txt_files(root):
    txt_files = []

    for dirname, _, filenames in os.walk(root):
        for filename in filenames:
            if filename.lower().endswith(".txt"):
                full_path = os.path.join(dirname, filename)
                txt_files.append(full_path)

    return sorted(txt_files)


txt_files = find_bonn_txt_files(DATA_ROOT)

print("=" * 70)
print("DATA DISCOVERY")
print("=" * 70)
print(f"Total .txt/.TXT files found: {len(txt_files)}")

print("\nFirst 10 files:")
for path in txt_files[:10]:
    print(path)


def infer_label_from_path(path):
    parts = path.replace("\\", "/").split("/")

    for part in parts:
        if part.upper() in ["F", "N", "O", "S", "Z"]:
            return part.upper()

    filename = os.path.basename(path)
    first_char = filename[0].upper()

    if first_char in ["F", "N", "O", "S", "Z"]:
        return first_char

    raise ValueError(f"Could not infer label from path: {path}")


X_full = []
y_labels = []

for path in txt_files:
    label = infer_label_from_path(path)
    signal = np.loadtxt(path, dtype=np.float32)

    X_full.append(signal)
    y_labels.append(label)

X_full = np.array(X_full, dtype=np.float32)
y_labels = np.array(y_labels)

class_names = ["F", "N", "O", "S", "Z"]
label_to_idx = {label: idx for idx, label in enumerate(class_names)}
idx_to_label = {idx: label for label, idx in label_to_idx.items()}

y_full = np.array([label_to_idx[label] for label in y_labels], dtype=np.int64)

print("\n" + "=" * 70)
print("LOADED DATA")
print("=" * 70)
print(f"X_full shape: {X_full.shape}")
print(f"y_full shape: {y_full.shape}")

if len(X_full) > 0:
    print(f"Signal length: {X_full.shape[1]}")

print("\nClass distribution:")
for idx, name in enumerate(class_names):
    count = np.sum(y_full == idx)
    print(f"{name}: {count}")

In [ ]:
# =========================================================
# CELL 4: FULL-SIGNAL TRAIN / VAL / TEST SPLIT
# =========================================================

# First split: 80% train+val, 20% test
X_trainval_full, X_test_full, y_trainval_full, y_test_full = train_test_split(
    X_full,
    y_full,
    test_size=0.20,
    random_state=SEED,
    stratify=y_full
)

# Second split: from remaining 80%, take 12.5% as val
# This gives 70% train, 10% val, 20% test overall
X_train_full, X_val_full, y_train_full, y_val_full = train_test_split(
    X_trainval_full,
    y_trainval_full,
    test_size=0.125,
    random_state=SEED,
    stratify=y_trainval_full
)

print("=" * 70)
print("FULL-SIGNAL SPLIT")
print("=" * 70)
print(f"Train signals: {X_train_full.shape[0]}")
print(f"Val signals:   {X_val_full.shape[0]}")
print(f"Test signals:  {X_test_full.shape[0]}")

print("\nClass distribution per split:")

for split_name, y_split in [
    ("Train", y_train_full),
    ("Val", y_val_full),
    ("Test", y_test_full),
]:
    print(f"\n{split_name}:")
    for idx, name in enumerate(class_names):
        print(f"  {name}: {np.sum(y_split == idx)}")

In [ ]:
# =========================================================
# CELL 5: CHUNK FULL SIGNALS
# =========================================================

def chunk_signals(X_signals, y_signals, chunk_size=256, stride=128):
    X_chunks = []
    y_chunks = []
    signal_ids = []

    for signal_idx, (signal, label) in enumerate(zip(X_signals, y_signals)):
        signal_len = len(signal)

        for start in range(0, signal_len - chunk_size + 1, stride):
            end = start + chunk_size
            chunk = signal[start:end]

            X_chunks.append(chunk)
            y_chunks.append(label)
            signal_ids.append(signal_idx)

    X_chunks = np.array(X_chunks, dtype=np.float32)
    y_chunks = np.array(y_chunks, dtype=np.int64)
    signal_ids = np.array(signal_ids, dtype=np.int64)

    return X_chunks, y_chunks, signal_ids


X_train_chunks, y_train_chunks, train_signal_ids = chunk_signals(
    X_train_full, y_train_full, chunk_size=chunk_size, stride=stride
)

X_val_chunks, y_val_chunks, val_signal_ids = chunk_signals(
    X_val_full, y_val_full, chunk_size=chunk_size, stride=stride
)

X_test_chunks, y_test_chunks, test_signal_ids = chunk_signals(
    X_test_full, y_test_full, chunk_size=chunk_size, stride=stride
)

print("=" * 70)
print("CHUNKING")
print("=" * 70)
print(f"X_train_chunks: {X_train_chunks.shape}")
print(f"X_val_chunks:   {X_val_chunks.shape}")
print(f"X_test_chunks:  {X_test_chunks.shape}")

print(f"\ny_train_chunks: {y_train_chunks.shape}")
print(f"y_val_chunks:   {y_val_chunks.shape}")
print(f"y_test_chunks:  {y_test_chunks.shape}")

print("\nChunks per signal:")
print(f"Train: {len(X_train_chunks) // len(X_train_full)}")
print(f"Val:   {len(X_val_chunks) // len(X_val_full)}")
print(f"Test:  {len(X_test_chunks) // len(X_test_full)}")

print("\nClass distribution in chunks:")
for split_name, y_split in [
    ("Train", y_train_chunks),
    ("Val", y_val_chunks),
    ("Test", y_test_chunks),
]:
    print(f"\n{split_name}:")
    for idx, name in enumerate(class_names):
        print(f"  {name}: {np.sum(y_split == idx)}")

In [ ]:
# =========================================================
# CELL 6: TRAIN-ONLY NORMALIZATION
# =========================================================

train_mean = X_train_chunks.mean()
train_std = X_train_chunks.std() + 1e-8

X_train_norm = (X_train_chunks - train_mean) / train_std
X_val_norm = (X_val_chunks - train_mean) / train_std
X_test_norm = (X_test_chunks - train_mean) / train_std

print("=" * 70)
print("TRAIN-ONLY NORMALIZATION")
print("=" * 70)
print(f"Train mean used: {train_mean:.6f}")
print(f"Train std used:  {train_std:.6f}")

print("\nAfter normalization:")
print(f"Train mean/std: {X_train_norm.mean():.4f} / {X_train_norm.std():.4f}")
print(f"Val mean/std:   {X_val_norm.mean():.4f} / {X_val_norm.std():.4f}")
print(f"Test mean/std:  {X_test_norm.mean():.4f} / {X_test_norm.std():.4f}")

print("\nShapes:")
print(f"X_train_norm: {X_train_norm.shape}")
print(f"X_val_norm:   {X_val_norm.shape}")
print(f"X_test_norm:  {X_test_norm.shape}")

In [ ]:
# =========================================================
# CELL 7: PURE DELTA-SIGMA SPIKE ENCODING
# =========================================================

def delta_sigma_encode_numpy(X, threshold=0.20):
    """
    Converts normalized EEG chunks into 2-channel delta-sigma spike trains.

    Input:
        X shape: [num_chunks, time]

    Output:
        spikes shape: [num_chunks, time, 2]
            channel 0 = positive delta-sigma spikes
            channel 1 = negative delta-sigma spikes
    """
    num_chunks, time_steps = X.shape

    spikes = np.zeros((num_chunks, time_steps, 2), dtype=np.float32)

    delta = np.diff(X, axis=1, prepend=X[:, :1])

    residual = np.zeros((num_chunks,), dtype=np.float32)

    for t in range(time_steps):
        residual += delta[:, t]

        pos_mask = residual >= threshold
        neg_mask = residual <= -threshold

        spikes[pos_mask, t, 0] = 1.0
        spikes[neg_mask, t, 1] = 1.0

        residual[pos_mask] -= threshold
        residual[neg_mask] += threshold

    return spikes


X_train_spike = delta_sigma_encode_numpy(X_train_norm, threshold=input_spike_threshold)
X_val_spike = delta_sigma_encode_numpy(X_val_norm, threshold=input_spike_threshold)
X_test_spike = delta_sigma_encode_numpy(X_test_norm, threshold=input_spike_threshold)

print("=" * 70)
print("PURE DELTA-SIGMA SPIKE ENCODING")
print("=" * 70)
print(f"Threshold: {input_spike_threshold}")

print("\nShapes:")
print(f"X_train_spike: {X_train_spike.shape}")
print(f"X_val_spike:   {X_val_spike.shape}")
print(f"X_test_spike:  {X_test_spike.shape}")

def spike_stats(name, X_spike):
    pos_rate = X_spike[:, :, 0].mean()
    neg_rate = X_spike[:, :, 1].mean()
    mean_activity = X_spike.mean()
    any_event_rate = (X_spike.sum(axis=2) > 0).mean()

    print(f"\n{name}:")
    print(f"  Positive spike rate:       {pos_rate:.4f}")
    print(f"  Negative spike rate:       {neg_rate:.4f}")
    print(f"  Mean channel activity:     {mean_activity:.4f}")
    print(f"  Any-event timestep rate:   {any_event_rate:.4f}")
    print(f"  Input sparsity:            {1.0 - mean_activity:.4f}")

spike_stats("Train", X_train_spike)
spike_stats("Val", X_val_spike)
spike_stats("Test", X_test_spike)

In [ ]:
# =========================================================
# CELL 8: SPIKE TENSOR DATASETS + DATALOADERS
# =========================================================

X_train_tensor = torch.tensor(X_train_spike, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_chunks, dtype=torch.long)

X_val_tensor = torch.tensor(X_val_spike, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val_chunks, dtype=torch.long)

X_test_tensor = torch.tensor(X_test_spike, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test_chunks, dtype=torch.long)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

print("=" * 70)
print("DATALOADERS")
print("=" * 70)
print(f"Train batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")
print(f"Test batches:  {len(test_loader)}")

xb, yb = next(iter(train_loader))
print("\nOne batch:")
print(f"xb shape: {xb.shape}")
print(f"yb shape: {yb.shape}")
print(f"xb dtype: {xb.dtype}")
print(f"yb dtype: {yb.dtype}")

In [ ]:
# =========================================================
# CELL 9: STRICT PURE MULTI-SCALE POOLED SLSTM - NO CONV LAYERS
# Corrected branches:
#   Branch 1 = any event in 4 samples
#   Branch 2 = repeated event in 4 samples
#   Branch 3 = burst event in 4 samples
# =========================================================

class FastSurrogateSpike(torch.autograd.Function):
    @staticmethod
    def forward(ctx, membrane, threshold):
        ctx.save_for_backward(membrane)
        ctx.threshold = threshold
        return (membrane >= threshold).float()

    @staticmethod
    def backward(ctx, grad_output):
        (membrane,) = ctx.saved_tensors
        threshold = ctx.threshold

        alpha = 5.0
        grad = grad_output / (alpha * torch.abs(membrane - threshold) + 1.0) ** 2

        return grad, None


class SpikeActivation(nn.Module):
    def __init__(self, threshold=0.20):
        super().__init__()
        self.threshold = threshold

    def forward(self, x):
        return FastSurrogateSpike.apply(x, self.threshold)




class SpikeDropout(nn.Module):
    """Drops spikes without inverted scaling, so activations remain binary."""
    def __init__(self, p=0.0):
        super().__init__()
        self.p = float(p)

    def forward(self, x):
        if not self.training or self.p <= 0.0:
            return x
        keep = torch.rand_like(x) >= self.p
        return x * keep.to(dtype=x.dtype)


class SpikingLSTMCell(nn.Module):
    def __init__(self, input_size, hidden_size, spike_threshold=0.12):
        super().__init__()

        self.input_size = input_size
        self.hidden_size = hidden_size
        self.spike_threshold = spike_threshold

        self.x2h = nn.Linear(input_size, 4 * hidden_size, bias=True)
        self.h2h = nn.Linear(hidden_size, 4 * hidden_size, bias=False)

        self.reset_parameters()

    def reset_parameters(self):
        nn.init.xavier_uniform_(self.x2h.weight)
        nn.init.xavier_uniform_(self.h2h.weight)
        nn.init.zeros_(self.x2h.bias)

    def forward(self, x_t, h_prev, c_prev):
        gates = self.x2h(x_t) + self.h2h(h_prev)

        i, f, g, o = gates.chunk(4, dim=1)

        i = torch.sigmoid(i)
        f = torch.sigmoid(f)
        g = torch.tanh(g)
        o = torch.sigmoid(o)

        c_t = f * c_prev + i * g
        membrane = o * torch.tanh(c_t)

        h_t = FastSurrogateSpike.apply(membrane, self.spike_threshold)

        return h_t, c_t


class StrictPureMultiScalePooledSLSTM(nn.Module):
    def __init__(
        self,
        input_channels=2,
        feature_dim=128,
        hidden1=192,
        hidden2=128,
        num_classes=5,
        dropout_p=0.12,
        projection_spike_threshold=0.20,
        hidden_spike_threshold=0.12
    ):
        super().__init__()

        self.hidden1 = hidden1
        self.hidden2 = hidden2

        self.projection_spike_threshold = projection_spike_threshold
        self.hidden_spike_threshold = hidden_spike_threshold

        # No Conv1D.
        # Event-only frontend:
        #   rate_pool4 gives coarse event density over 4 samples.
        #   rate_pool2 gives early/late event timing inside each 4-sample window.
        self.rate_pool2 = nn.AvgPool1d(kernel_size=2, stride=2)
        self.rate_pool4 = nn.AvgPool1d(kernel_size=4, stride=4)

        # Coarse density branches from 4-sample windows:
        # threshold 0.25 => at least 1 spike in 4 samples
        # threshold 0.50 => at least 2 spikes in 4 samples
        # threshold 0.75 => at least 3 spikes in 4 samples
        self.any_event_spike = SpikeActivation(threshold=0.25)
        self.repeated_event_spike = SpikeActivation(threshold=0.50)
        self.burst_event_spike = SpikeActivation(threshold=0.75)

        # Fine timing branch: at least 1 spike in each 2-sample half-window.
        self.half_window_event_spike = SpikeActivation(threshold=0.50)

        # Concatenated branches:
        #   3 coarse branches x 2 polarities = 6
        #   2 half-window timing branches x 2 polarities = 4
        # Total = 10 event features per reduced timestep.
        multiscale_input_dim = input_channels * 5

        self.input_project = nn.Sequential(
            nn.Linear(multiscale_input_dim, feature_dim),
            nn.LayerNorm(feature_dim),
            SpikeActivation(threshold=projection_spike_threshold),
            SpikeDropout(dropout_p)
        )

        self.slstm1 = SpikingLSTMCell(
            input_size=feature_dim,
            hidden_size=hidden1,
            spike_threshold=hidden_spike_threshold
        )

        self.slstm2 = SpikingLSTMCell(
            input_size=hidden1,
            hidden_size=hidden2,
            spike_threshold=hidden_spike_threshold
        )

        self.attn = nn.Sequential(
            nn.Linear(hidden2, hidden2),
            nn.Tanh(),
            nn.Linear(hidden2, 1)
        )

        # Spiking readout MLP; final linear layer produces logits.
        self.classifier = nn.Sequential(
            nn.Linear(hidden2, hidden2),
            nn.LayerNorm(hidden2),
            SpikeActivation(threshold=hidden_spike_threshold),
            SpikeDropout(dropout_p),
            nn.Linear(hidden2, num_classes)
        )

    def forward(self, x, return_spike_stats=False, return_attention=False):
        """
        x shape: [B, 256, 2]
        """
        B, T, C = x.shape

        input_activity = x.mean()

        # Pooling expects [B, C, T]
        x_t = x.transpose(1, 2)

        # Coarse event density over 4-sample windows.
        rate4 = self.rate_pool4(x_t)  # [B, 2, 64]

        # Fine event timing over 2-sample half-windows, then regroup to 64 steps.
        rate2 = self.rate_pool2(x_t)  # [B, 2, 128]
        half_events = self.half_window_event_spike(rate2)
        half_events = half_events.reshape(B, C, rate4.size(-1), 2)
        early_event = half_events[:, :, :, 0]  # [B, 2, 64]
        late_event = half_events[:, :, :, 1]   # [B, 2, 64]

        # Three coarse spike summaries.
        b1 = self.any_event_spike(rate4)       # at least 1 spike in 4 samples
        b2 = self.repeated_event_spike(rate4)  # at least 2 spikes in 4 samples
        b3 = self.burst_event_spike(rate4)     # at least 3 spikes in 4 samples

        b1_activity = b1.mean()
        b2_activity = b2.mean()
        b3_activity = b3.mean()
        half_activity = torch.cat([early_event, late_event], dim=1).mean()

        z = torch.cat([b1, b2, b3, early_event, late_event], dim=1)  # [B, 10, 64]
        multiscale_activity = z.mean()

        z = z.transpose(1, 2)                  # [B, 64, 10]

        z = self.input_project(z)              # [B, 64, feature_dim]
        projected_activity = z.mean()

        B, T_reduced, _ = z.shape

        h1 = torch.zeros(B, self.hidden1, device=x.device)
        c1 = torch.zeros(B, self.hidden1, device=x.device)

        h2 = torch.zeros(B, self.hidden2, device=x.device)
        c2 = torch.zeros(B, self.hidden2, device=x.device)

        outputs = []
        h1_spikes = []
        h2_spikes = []

        for t in range(T_reduced):
            h1, c1 = self.slstm1(z[:, t, :], h1, c1)
            h2, c2 = self.slstm2(h1, h2, c2)

            outputs.append(h2.unsqueeze(1))

            if return_spike_stats:
                h1_spikes.append(h1.detach())
                h2_spikes.append(h2.detach())

        H = torch.cat(outputs, dim=1)

        attn_scores = self.attn(H).squeeze(-1)
        attn_weights = torch.softmax(attn_scores, dim=1).unsqueeze(-1)

        pooled = torch.sum(H * attn_weights, dim=1)

        logits = self.classifier(pooled)

        if return_spike_stats:
            h1_rate = torch.stack(h1_spikes, dim=1).mean()
            h2_rate = torch.stack(h2_spikes, dim=1).mean()
            hidden_spike_rate = (h1_rate + h2_rate) / 2.0

            stats = {
                "input_activity": input_activity.detach(),
                "branch1_any_event_activity": b1_activity.detach(),
                "branch2_repeated_event_activity": b2_activity.detach(),
                "branch3_burst_event_activity": b3_activity.detach(),
                "half_window_event_activity": half_activity.detach(),
                "multiscale_activity": multiscale_activity.detach(),
                "projected_activity": projected_activity.detach(),
                "hidden_spike_rate": hidden_spike_rate.detach(),
                "h1_spike_rate": h1_rate.detach(),
                "h2_spike_rate": h2_rate.detach(),
                "reduced_timesteps": torch.tensor(T_reduced, device=x.device)
            }

            if return_attention:
                stats["attention_weights"] = attn_weights.squeeze(-1).detach()

            return logits, stats

        if return_attention:
            return logits, {"attention_weights": attn_weights.squeeze(-1).detach()}

        return logits


# Optimized strict pure no-conv config
feature_dim = 128
hidden1 = 192
hidden2 = 128

projection_spike_threshold = 0.20
hidden_spike_threshold = 0.12
dropout_p = 0.12

model = StrictPureMultiScalePooledSLSTM(
    input_channels=input_channels,
    feature_dim=feature_dim,
    hidden1=hidden1,
    hidden2=hidden2,
    num_classes=num_classes,
    dropout_p=dropout_p,
    projection_spike_threshold=projection_spike_threshold,
    hidden_spike_threshold=hidden_spike_threshold
).to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("=" * 70)
print("STRICT PURE MULTI-SCALE POOLED SLSTM - NO CONV LAYERS")
print("=" * 70)
print(f"feature_dim:                  {feature_dim}")
print(f"hidden sizes:                 {hidden1} -> {hidden2}")
print(f"projection_spike_threshold:   {projection_spike_threshold}")
print(f"hidden_spike_threshold:       {hidden_spike_threshold}")
print(f"dropout_p:                    {dropout_p}")
print(model)
print(f"\nTrainable parameters: {total_params:,}")


In [ ]:
# =========================================================
# CELL 10: SANITY FORWARD PASS
# =========================================================

model.eval()

xb, yb = next(iter(train_loader))
xb = xb.to(device)
yb = yb.to(device)

with torch.no_grad():
    logits, stats = model(xb, return_spike_stats=True)

print("=" * 70)
print("SANITY FORWARD PASS")
print("=" * 70)
print(f"Input batch shape:  {xb.shape}")
print(f"Logits shape:       {logits.shape}")
print(f"Targets shape:      {yb.shape}")

print("\nSpike/activity stats:")
for k, v in stats.items():
    print(f"{k}: {v.item():.4f}")

print("\nSample logits:")
print(logits[:3])

print("\nSample predicted classes:")
print(torch.argmax(logits[:10], dim=1).detach().cpu().numpy())

print("\nSample true classes:")
print(yb[:10].detach().cpu().numpy())

In [ ]:
# =========================================================
# CELL 11: TRAIN / EVALUATE FUNCTIONS FOR STRICT PURE MULTI-SCALE MODEL
# =========================================================

criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate,
    weight_decay=weight_decay
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=epochs,
    eta_min=1e-5
)


def apply_spike_time_shift(x, max_shift=0):
    if max_shift <= 0:
        return x

    B = x.size(0)
    shifts = torch.randint(
        low=-max_shift,
        high=max_shift + 1,
        size=(B,),
        device=x.device
    )

    shifted = torch.zeros_like(x)

    for i, shift in enumerate(shifts.tolist()):
        if shift > 0:
            shifted[i, shift:, :] = x[i, :-shift, :]
        elif shift < 0:
            shifted[i, :shift, :] = x[i, -shift:, :]
        else:
            shifted[i] = x[i]

    return shifted


def augment_spike_batch(x, time_shift_max=0, spike_dropout_p=0.0):
    x = apply_spike_time_shift(x, max_shift=time_shift_max)

    if spike_dropout_p > 0.0:
        keep = torch.rand_like(x) >= spike_dropout_p
        x = x * keep.to(dtype=x.dtype)

    return x


def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()

    running_loss = 0.0
    all_preds = []
    all_targets = []

    input_acts = []
    branch1_acts = []
    branch2_acts = []
    branch3_acts = []
    multiscale_acts = []
    proj_acts = []
    hidden_spks = []

    loop = tqdm(loader, desc="Training", leave=False)

    for xb, yb in loop:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)

        xb_aug = augment_spike_batch(
            xb,
            time_shift_max=train_time_shift_max,
            spike_dropout_p=input_spike_dropout_p
        )

        optimizer.zero_grad(set_to_none=True)

        logits, stats = model(xb_aug, return_spike_stats=True)
        loss = criterion(logits, yb)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        running_loss += loss.item() * xb.size(0)

        preds = torch.argmax(logits, dim=1)

        all_preds.extend(preds.detach().cpu().numpy())
        all_targets.extend(yb.detach().cpu().numpy())

        input_acts.append(stats["input_activity"].item())
        branch1_acts.append(stats["branch1_any_event_activity"].item())
        branch2_acts.append(stats["branch2_repeated_event_activity"].item())
        branch3_acts.append(stats["branch3_burst_event_activity"].item())
        multiscale_acts.append(stats["multiscale_activity"].item())
        proj_acts.append(stats["projected_activity"].item())
        hidden_spks.append(stats["hidden_spike_rate"].item())

        loop.set_postfix({
            "loss": f"{loss.item():.4f}",
            "in": f"{stats['input_activity'].item():.3f}",
            "b1": f"{stats['branch1_any_event_activity'].item():.3f}",
            "b2": f"{stats['branch2_repeated_event_activity'].item():.3f}",
            "b3": f"{stats['branch3_burst_event_activity'].item():.3f}",
            "hid": f"{stats['hidden_spike_rate'].item():.3f}"
        })

    epoch_loss = running_loss / len(loader.dataset)

    acc = accuracy_score(all_targets, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_targets,
        all_preds,
        average="macro",
        zero_division=0
    )

    return {
        "loss": epoch_loss,
        "acc": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "input_activity": float(np.mean(input_acts)),
        "branch1_activity": float(np.mean(branch1_acts)),
        "branch2_activity": float(np.mean(branch2_acts)),
        "branch3_activity": float(np.mean(branch3_acts)),
        "multiscale_activity": float(np.mean(multiscale_acts)),
        "projected_activity": float(np.mean(proj_acts)),
        "hidden_spike_rate": float(np.mean(hidden_spks))
    }


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()

    running_loss = 0.0
    all_preds = []
    all_targets = []

    input_acts = []
    branch1_acts = []
    branch2_acts = []
    branch3_acts = []
    multiscale_acts = []
    proj_acts = []
    hidden_spks = []

    for xb, yb in loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)

        logits, stats = model(xb, return_spike_stats=True)
        loss = criterion(logits, yb)

        running_loss += loss.item() * xb.size(0)

        preds = torch.argmax(logits, dim=1)

        all_preds.extend(preds.detach().cpu().numpy())
        all_targets.extend(yb.detach().cpu().numpy())

        input_acts.append(stats["input_activity"].item())
        branch1_acts.append(stats["branch1_any_event_activity"].item())
        branch2_acts.append(stats["branch2_repeated_event_activity"].item())
        branch3_acts.append(stats["branch3_burst_event_activity"].item())
        multiscale_acts.append(stats["multiscale_activity"].item())
        proj_acts.append(stats["projected_activity"].item())
        hidden_spks.append(stats["hidden_spike_rate"].item())

    epoch_loss = running_loss / len(loader.dataset)

    acc = accuracy_score(all_targets, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_targets,
        all_preds,
        average="macro",
        zero_division=0
    )

    return {
        "loss": epoch_loss,
        "acc": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "input_activity": float(np.mean(input_acts)),
        "branch1_activity": float(np.mean(branch1_acts)),
        "branch2_activity": float(np.mean(branch2_acts)),
        "branch3_activity": float(np.mean(branch3_acts)),
        "multiscale_activity": float(np.mean(multiscale_acts)),
        "projected_activity": float(np.mean(proj_acts)),
        "hidden_spike_rate": float(np.mean(hidden_spks)),
        "preds": np.array(all_preds),
        "targets": np.array(all_targets)
    }


print("=" * 70)
print("TRAINING SETUP READY - STRICT PURE MULTI-SCALE NO-CONV MODEL")
print("=" * 70)
print(f"Criterion: CrossEntropyLoss(label_smoothing={label_smoothing})")
print(f"Optimizer: AdamW(lr={learning_rate}, weight_decay={weight_decay})")
print(f"Scheduler: CosineAnnealingLR(T_max={epochs}, eta_min=1e-5)")
print(f"Spike augmentation: shift +/-{train_time_shift_max}, input spike dropout {input_spike_dropout_p}")

# =========================================================
# SIGNAL-LEVEL VALIDATION + PDF FIGURE HELPERS
# =========================================================

def save_pdf_figure(filename):
    """
    Save the current matplotlib figure as a PDF in FIG_DIR.
    """
    path = os.path.join(FIG_DIR, filename)
    plt.savefig(path, format="pdf", bbox_inches="tight")
    print(f"Saved PDF: {path}")


def chunk_one_signal(signal, chunk_size=256, stride=128):
    chunks = []

    for start in range(0, len(signal) - chunk_size + 1, stride):
        end = start + chunk_size
        chunks.append(signal[start:end])

    return np.array(chunks, dtype=np.float32)


def signal_to_spike_tensor(signal):
    """
    Uses the same train-only normalization and delta-sigma encoder as training.
    """
    chunks = chunk_one_signal(signal, chunk_size=chunk_size, stride=stride)
    chunks_norm = (chunks - train_mean) / train_std
    chunks_spike = delta_sigma_encode_numpy(
        chunks_norm,
        threshold=input_spike_threshold
    )

    return torch.tensor(chunks_spike, dtype=torch.float32).to(device), chunks_spike


@torch.no_grad()
def evaluate_signal_level(model, X_signals, y_signals, criterion=None, aggregation="avg_probs"):
    """
    Signal-level-only evaluation.

    aggregation:
      - "majority"
      - "avg_logits"
      - "avg_probs"
    """
    model.eval()

    signal_targets = []
    signal_preds = []

    signal_losses = []

    input_acts = []
    branch1_acts = []
    branch2_acts = []
    branch3_acts = []
    multiscale_acts = []
    proj_acts = []
    hidden_spks = []

    for signal, label in zip(X_signals, y_signals):
        chunks_tensor, _ = signal_to_spike_tensor(signal)

        logits, stats = model(chunks_tensor, return_spike_stats=True)
        probs = torch.softmax(logits, dim=1)
        chunk_preds = torch.argmax(probs, dim=1).detach().cpu().numpy()

        if aggregation == "majority":
            pred = np.bincount(chunk_preds, minlength=num_classes).argmax()
        elif aggregation == "avg_logits":
            pred = torch.argmax(logits.mean(dim=0)).item()
        elif aggregation == "avg_probs":
            pred = torch.argmax(probs.mean(dim=0)).item()
        else:
            raise ValueError("aggregation must be one of: majority, avg_logits, avg_probs")

        if criterion is not None:
            y_repeated = torch.full(
                (logits.size(0),),
                int(label),
                dtype=torch.long,
                device=logits.device
            )
            signal_losses.append(criterion(logits, y_repeated).item())

        signal_targets.append(int(label))
        signal_preds.append(int(pred))

        input_acts.append(stats["input_activity"].item())
        branch1_acts.append(stats["branch1_any_event_activity"].item())
        branch2_acts.append(stats["branch2_repeated_event_activity"].item())
        branch3_acts.append(stats["branch3_burst_event_activity"].item())
        multiscale_acts.append(stats["multiscale_activity"].item())
        proj_acts.append(stats["projected_activity"].item())
        hidden_spks.append(stats["hidden_spike_rate"].item())

    signal_targets = np.array(signal_targets)
    signal_preds = np.array(signal_preds)

    acc = accuracy_score(signal_targets, signal_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        signal_targets,
        signal_preds,
        average="macro",
        zero_division=0
    )

    return {
        "loss": float(np.mean(signal_losses)) if len(signal_losses) > 0 else np.nan,
        "acc": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "input_activity": float(np.mean(input_acts)),
        "branch1_activity": float(np.mean(branch1_acts)),
        "branch2_activity": float(np.mean(branch2_acts)),
        "branch3_activity": float(np.mean(branch3_acts)),
        "multiscale_activity": float(np.mean(multiscale_acts)),
        "projected_activity": float(np.mean(proj_acts)),
        "hidden_spike_rate": float(np.mean(hidden_spks)),
        "preds": signal_preds,
        "targets": signal_targets
    }



def _overlap_add_1d(values_by_chunk, signal_len, chunk_size=256, stride=128):
    timeline = np.zeros(signal_len, dtype=np.float32)
    counts = np.zeros(signal_len, dtype=np.float32)

    for chunk_idx, values in enumerate(values_by_chunk):
        start = chunk_idx * stride
        end = min(start + chunk_size, signal_len)
        usable = end - start

        timeline[start:end] += values[:usable]
        counts[start:end] += 1.0

    return timeline / np.maximum(counts, 1.0)


def _normalize_01(values):
    values = np.asarray(values, dtype=np.float32)
    v_min = float(np.min(values))
    v_max = float(np.max(values))

    if v_max <= v_min:
        return np.zeros_like(values)

    return (values - v_min) / (v_max - v_min)


@torch.no_grad()
def _signal_dashboard_data(signal, label):
    chunks_tensor, chunks_spike = signal_to_spike_tensor(signal)

    logits, stats = model(
        chunks_tensor,
        return_spike_stats=True,
        return_attention=True
    )

    probs = torch.softmax(logits, dim=1).detach().cpu().numpy()
    chunk_preds = probs.argmax(axis=1)
    chunk_conf = probs.max(axis=1)
    signal_probs = probs.mean(axis=0)
    signal_pred = int(signal_probs.argmax())

    attn = stats["attention_weights"].detach().cpu().numpy()
    attn_expanded = np.repeat(attn, repeats=chunk_size // attn.shape[1], axis=1)
    attn_timeline = _overlap_add_1d(
        attn_expanded,
        signal_len=len(signal),
        chunk_size=chunk_size,
        stride=stride
    )

    spike_any = (chunks_spike.sum(axis=2) > 0).astype(np.float32)
    spike_density = _overlap_add_1d(
        spike_any,
        signal_len=len(signal),
        chunk_size=chunk_size,
        stride=stride
    )

    return {
        "chunks_tensor": chunks_tensor,
        "chunks_spike": chunks_spike,
        "probs": probs,
        "chunk_preds": chunk_preds,
        "chunk_conf": chunk_conf,
        "signal_probs": signal_probs,
        "signal_pred": signal_pred,
        "attn_timeline": attn_timeline,
        "spike_density": spike_density,
        "label": int(label),
    }


@torch.no_grad()
def _chunk_occlusion_drop(chunks_tensor, target_class):
    base_logits = model(chunks_tensor)
    base_probs = torch.softmax(base_logits, dim=1).mean(dim=0)
    base_prob = base_probs[target_class].item()

    drops = []
    for chunk_idx in range(chunks_tensor.size(0)):
        masked = chunks_tensor.clone()
        masked[chunk_idx] = 0.0

        masked_logits = model(masked)
        masked_probs = torch.softmax(masked_logits, dim=1).mean(dim=0)
        drop = base_prob - masked_probs[target_class].item()
        drops.append(drop)

    return np.array(drops, dtype=np.float32)


def plot_signal_diagnostic_dashboard(
    stage_name,
    signal=None,
    label=None,
    signal_index=0,
    compute_occlusion=False,
    aggregation="avg_probs"
):
    """
    Minimal signal-level diagnostic:
      1. final class probabilities
      2. class probabilities across chunks
      3. optional chunk occlusion impact
    """
    model.eval()

    if signal is None:
        signal = X_val_full[signal_index]
        label = y_val_full[signal_index]

    data = _signal_dashboard_data(signal, label)

    chunk_count = data["probs"].shape[0]
    chunk_idx = np.arange(chunk_count)

    true_label = data["label"]
    pred_label = data["signal_pred"]
    pred_prob = data["signal_probs"][pred_label]

    nrows = 3 if compute_occlusion else 2
    height = 5.8 if compute_occlusion else 4.6
    ratios = [1.0, 2.2, 1.0] if compute_occlusion else [1.0, 2.2]

    fig, axes = plt.subplots(
        nrows=nrows,
        ncols=1,
        figsize=(9.5, height),
        gridspec_kw={"height_ratios": ratios},
        constrained_layout=True
    )

    if nrows == 2:
        ax_prob, ax_chunks = axes
        ax_occ = None
    else:
        ax_prob, ax_chunks, ax_occ = axes

    # 1) Final signal-level probabilities.
    colors = ["#888888"] * num_classes
    colors[true_label] = "#2ca02c"
    if pred_label != true_label:
        colors[pred_label] = "#d62728"

    ax_prob.barh(np.arange(num_classes), data["signal_probs"], color=colors, alpha=0.9)
    ax_prob.set_yticks(np.arange(num_classes))
    ax_prob.set_yticklabels(class_names)
    ax_prob.set_xlim(0, 1)
    ax_prob.invert_yaxis()
    ax_prob.set_xlabel("Average probability")
    ax_prob.set_title(
        f"{stage_name} | true: {class_names[true_label]} | "
        f"predicted: {class_names[pred_label]} ({pred_prob:.2f})"
    )

    for class_idx, value in enumerate(data["signal_probs"]):
        ax_prob.text(
            min(float(value) + 0.02, 0.98),
            class_idx,
            f"{value:.2f}",
            va="center",
            fontsize=9
        )

    # 2) Chunk probability heatmap.
    heatmap = ax_chunks.imshow(
        data["probs"].T,
        aspect="auto",
        interpolation="nearest",
        cmap="Blues",
        vmin=0,
        vmax=1
    )
    ax_chunks.set_yticks(np.arange(num_classes))
    ax_chunks.set_yticklabels(class_names)
    ax_chunks.set_ylabel("Class")
    ax_chunks.set_xlabel("Chunk index")
    ax_chunks.set_title("Evidence across chunks")

    cbar = fig.colorbar(heatmap, ax=ax_chunks, fraction=0.025, pad=0.015)
    cbar.set_label("prob.", rotation=270, labelpad=11)

    # 3) Optional causal chunk importance.
    if compute_occlusion:
        drops = _chunk_occlusion_drop(data["chunks_tensor"], pred_label)
        ax_occ.bar(chunk_idx, drops, color="#4c78a8", alpha=0.9)
        ax_occ.axhline(0, color="black", linewidth=0.8)
        ax_occ.set_xlim(-0.5, chunk_count - 0.5)
        ax_occ.set_xlabel("Chunk index")
        ax_occ.set_ylabel("Drop")
        ax_occ.set_title("Occlusion impact on predicted class")

    safe_stage = stage_name.lower().replace(" ", "_").replace("/", "_")
    save_pdf_figure(f"signal_dashboard_{safe_stage}.pdf")
    plt.show()

    return data

def plot_attention_map_for_signal(stage_name, signal=None, label=None, signal_index=0):
    """
    Backward-compatible wrapper used by the training loop.
    Skips the pre-training plot because untrained probabilities are visually noisy.
    """
    stage = stage_name.lower()
    if stage.startswith("before training") or stage.startswith("during training"):
        print(f"Skipping {stage_name} dashboard; minimal dashboards are saved after training and on test signals.")
        return None

    return plot_signal_diagnostic_dashboard(
        stage_name=stage_name,
        signal=signal,
        label=label,
        signal_index=signal_index,
        compute_occlusion=False
    )


@torch.no_grad()
def plot_confident_mistake_gallery(X_signals, y_signals, top_n=3, aggregation="avg_probs", plot_examples=False):
    mistakes = []

    for signal_idx, (signal, label) in enumerate(zip(X_signals, y_signals)):
        data = _signal_dashboard_data(signal, label)
        pred = data["signal_pred"]

        if pred != int(label):
            mistakes.append({
                "signal_idx": signal_idx,
                "true": int(label),
                "pred": pred,
                "confidence": float(data["signal_probs"][pred]),
                "margin": float(
                    np.sort(data["signal_probs"])[-1] - np.sort(data["signal_probs"])[-2]
                )
            })

    if len(mistakes) == 0:
        print("No signal-level mistakes found for the selected split.")
        return pd.DataFrame(columns=["signal_idx", "true", "pred", "confidence", "margin"])

    mistake_df = pd.DataFrame(mistakes).sort_values(
        ["confidence", "margin"],
        ascending=False
    ).head(top_n)

    print("Most confident signal-level mistakes:")
    display(mistake_df.assign(
        true=mistake_df["true"].map(idx_to_label),
        pred=mistake_df["pred"].map(idx_to_label)
    ))

    if plot_examples:
        for _, row in mistake_df.iterrows():
            plot_signal_diagnostic_dashboard(
                stage_name=f"Confident Mistake Signal {int(row['signal_idx'])}",
                signal=X_signals[int(row["signal_idx"])],
                label=y_signals[int(row["signal_idx"])],
                compute_occlusion=False,
                aggregation=aggregation
            )

    return mistake_df


In [ ]:

# =========================================================
# CELL 12: TRAINING LOOP - FIXED EPOCHS, NO EARLY STOPPING
# Signal-level validation only.
# Also saves attention/input-spike maps before, during, and after training.
# =========================================================

best_val_f1 = -1.0
best_epoch = -1
best_state = None
final_state = None

history = []

mid_training_epoch = epochs // 2

print("=" * 70)
print("TRAINING START - FULL RUN, NO EARLY STOPPING")
print("=" * 70)
print("Validation metric is SIGNAL-LEVEL using average-probability aggregation.")
print(f"PDF figures folder: {FIG_DIR}")

# Before-training maps
plot_attention_map_for_signal(
    stage_name="Before Training",
    signal_index=0
)

for epoch in range(1, epochs + 1):
    start_time = time.time()
    current_lr = optimizer.param_groups[0]["lr"]

    print(f"\n===== Epoch {epoch:02d}/{epochs} | LR: {current_lr:.6f} =====")

    # Training still uses chunks because the model input is chunk-shaped.
    # Evaluation/reporting is signal-level only.
    train_metrics = train_one_epoch(
        model=model,
        loader=train_loader,
        criterion=criterion,
        optimizer=optimizer,
        device=device
    )

    val_signal_metrics = evaluate_signal_level(
        model=model,
        X_signals=X_val_full,
        y_signals=y_val_full,
        criterion=criterion,
        aggregation="avg_probs"
    )

    scheduler.step()

    elapsed = time.time() - start_time

    row = {
        "epoch": epoch,
        "lr": current_lr,
        "train_loss": train_metrics["loss"],
        "train_acc_chunk_internal": train_metrics["acc"],
        "train_f1_chunk_internal": train_metrics["f1"],
        "val_signal_loss": val_signal_metrics["loss"],
        "val_signal_acc": val_signal_metrics["acc"],
        "val_signal_f1": val_signal_metrics["f1"],
        "val_signal_input_activity": val_signal_metrics["input_activity"],
        "val_signal_multiscale_activity": val_signal_metrics["multiscale_activity"],
        "val_signal_projected_activity": val_signal_metrics["projected_activity"],
        "val_signal_hidden_spike_rate": val_signal_metrics["hidden_spike_rate"],
        "time_sec": elapsed
    }

    history.append(row)

    print(
        f"Epoch {epoch:02d} | "
        f"Train Loss {train_metrics['loss']:.4f} | "
        f"Val SIGNAL Loss {val_signal_metrics['loss']:.4f} "
        f"Acc {val_signal_metrics['acc']:.4f} "
        f"F1 {val_signal_metrics['f1']:.4f} | "
        f"In {val_signal_metrics['input_activity']:.4f} "
        f"MS {val_signal_metrics['multiscale_activity']:.4f} "
        f"Proj {val_signal_metrics['projected_activity']:.4f} "
        f"Hid {val_signal_metrics['hidden_spike_rate']:.4f} | "
        f"Time {elapsed:.1f}s"
    )

    # Checkpoint only; no early stopping and no automatic loading of best weights.
    if val_signal_metrics["f1"] > best_val_f1:
        best_val_f1 = val_signal_metrics["f1"]
        best_epoch = epoch
        best_state = copy.deepcopy(model.state_dict())
        print(f"  New best checkpoint saved. Best SIGNAL Val F1 = {best_val_f1:.4f} at epoch {best_epoch}")
    else:
        print("  No best-checkpoint update. Continuing full 60-epoch run.")

    if epoch == mid_training_epoch:
        plot_attention_map_for_signal(
            stage_name=f"During Training Epoch {epoch:02d}",
            signal_index=0
        )

final_state = copy.deepcopy(model.state_dict())

print("\n" + "=" * 70)
print("TRAINING COMPLETE — NO EARLY STOPPING USED")
print("=" * 70)
print(f"Completed epochs: {len(history)}/{epochs}")
print(f"Best SIGNAL Val F1 checkpoint: {best_val_f1:.4f}")
print(f"Best checkpoint epoch:         {best_epoch}")
print("Final epoch weights are saved separately in final_state.")

# After-training maps
plot_attention_map_for_signal(
    stage_name=f"After Training Epoch {epochs:02d}",
    signal_index=0
)

# Use the best signal-level validation checkpoint for all downstream test/FLOPs/energy cells.
# This keeps the fixed full run while avoiding evaluation on an over-trained final epoch.
if use_best_checkpoint_for_test and best_state is not None:
    model.load_state_dict(best_state)
    active_checkpoint_name = f"best validation checkpoint from epoch {best_epoch}"
else:
    active_checkpoint_name = f"final epoch checkpoint from epoch {epochs}"

print(f"Model weights now set to: {active_checkpoint_name}")

# Training history plots saved as PDF
history_df = pd.DataFrame(history)

plt.figure(figsize=(9, 5))
plt.plot(history_df["epoch"], history_df["train_loss"], label="Train loss")
plt.plot(history_df["epoch"], history_df["val_signal_loss"], label="Val signal loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Signal-Level Validation Loss")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
save_pdf_figure("training_signal_validation_loss.pdf")
plt.show()

plt.figure(figsize=(9, 5))
plt.plot(history_df["epoch"], history_df["val_signal_acc"], label="Val signal accuracy")
plt.plot(history_df["epoch"], history_df["val_signal_f1"], label="Val signal macro-F1")
plt.xlabel("Epoch")
plt.ylabel("Score")
plt.title("Signal-Level Validation Accuracy and Macro-F1")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
save_pdf_figure("signal_validation_accuracy_f1.pdf")
plt.show()

plt.figure(figsize=(9, 5))
plt.plot(history_df["epoch"], history_df["val_signal_input_activity"], label="Input spike rate")
plt.plot(history_df["epoch"], history_df["val_signal_multiscale_activity"], label="Multiscale spike rate")
plt.plot(history_df["epoch"], history_df["val_signal_projected_activity"], label="Projected spike rate")
plt.plot(history_df["epoch"], history_df["val_signal_hidden_spike_rate"], label="Hidden spike rate")
plt.xlabel("Epoch")
plt.ylabel("Average activity")
plt.title("Signal-Level Spike Activity During Training")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
save_pdf_figure("signal_spike_activity_over_epochs.pdf")
plt.show()


In [ ]:
# =========================================================
# COMPACT SIGNAL-LEVEL TEST EVALUATION ONLY
# Average-probability aggregation + most important metrics only
# Saves compact summary and confusion matrix as PDF
# =========================================================

import time
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix
)

model.eval()

test_aggregation = "avg_probs"
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

all_signal_times = []
all_chunks_per_signal = []

signal_targets = []
signal_preds_majority = []
signal_avg_probs = []

all_signal_input_activity = []
all_signal_hidden_spike_rate = []

# -----------------------------
# Warm-up pass
# -----------------------------
with torch.no_grad():
    warm_tensor, _ = signal_to_spike_tensor(X_test_full[0])
    _ = model(warm_tensor)

if torch.cuda.is_available():
    torch.cuda.synchronize()

start_total = time.time()

# -----------------------------
# Signal-level inference
# -----------------------------
with torch.no_grad():
    for signal, label in zip(X_test_full, y_test_full):
        chunks_tensor, _ = signal_to_spike_tensor(signal)

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        start_signal = time.time()
        logits, stats = model(chunks_tensor, return_spike_stats=True)

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        signal_time = time.time() - start_signal

        probs = torch.softmax(logits, dim=1)
        avg_probs = probs.mean(dim=0).detach().cpu().numpy()
        chunk_preds = torch.argmax(probs, dim=1).detach().cpu().numpy()

        if test_aggregation == "majority":
            signal_pred = np.bincount(
                chunk_preds,
                minlength=num_classes
            ).argmax()
        elif test_aggregation == "avg_logits":
            signal_pred = torch.argmax(logits.mean(dim=0)).item()
        elif test_aggregation == "avg_probs":
            signal_pred = torch.argmax(probs.mean(dim=0)).item()
        else:
            raise ValueError("test_aggregation must be one of: majority, avg_logits, avg_probs")

        signal_targets.append(int(label))
        signal_preds_majority.append(int(signal_pred))
        signal_avg_probs.append(avg_probs)

        all_signal_input_activity.append(stats["input_activity"].item())
        all_signal_hidden_spike_rate.append(stats["hidden_spike_rate"].item())

        all_signal_times.append(signal_time)
        all_chunks_per_signal.append(chunks_tensor.size(0))

total_signal_eval_time = time.time() - start_total

signal_targets = np.array(signal_targets)
signal_preds_majority = np.array(signal_preds_majority)
signal_avg_probs = np.vstack(signal_avg_probs)

# -----------------------------
# Main metrics
# -----------------------------
acc = accuracy_score(signal_targets, signal_preds_majority)

prec, rec, f1, _ = precision_recall_fscore_support(
    signal_targets,
    signal_preds_majority,
    average="macro",
    zero_division=0
)

class_prec, class_rec, class_f1, class_support = precision_recall_fscore_support(
    signal_targets,
    signal_preds_majority,
    average=None,
    labels=list(range(num_classes)),
    zero_division=0
)

avg_input_activity = float(np.mean(all_signal_input_activity))
avg_hidden_spike_rate = float(np.mean(all_signal_hidden_spike_rate))

avg_latency_per_signal = float(np.mean(all_signal_times))
avg_chunks_per_signal = float(np.mean(all_chunks_per_signal))
avg_latency_per_chunk = avg_latency_per_signal / avg_chunks_per_signal
signal_throughput = len(signal_targets) / total_signal_eval_time

# -----------------------------
# Compact summary table
# -----------------------------
summary_df = pd.DataFrame({
    "Metric": [
        "Signal Aggregation",
        "Active Checkpoint",
        "Signal Accuracy",
        "Signal Macro-F1",
        "Macro Precision",
        "Macro Recall",
        "Total Parameters",
        "Input Spike Rate",
        "Input Sparsity",
        "Hidden Spike Rate",
        "Hidden Sparsity",
        "Latency per Chunk",
        "Latency per Signal",
        "Signal Throughput",
        "Avg. Chunks per Signal"
    ],
    "Value": [
        test_aggregation,
        globals().get("active_checkpoint_name", "current model weights"),
        f"{acc:.4f}",
        f"{f1:.4f}",
        f"{prec:.4f}",
        f"{rec:.4f}",
        f"{total_params:,}",
        f"{avg_input_activity * 100:.2f}%",
        f"{(1.0 - avg_input_activity) * 100:.2f}%",
        f"{avg_hidden_spike_rate * 100:.2f}%",
        f"{(1.0 - avg_hidden_spike_rate) * 100:.2f}%",
        f"{avg_latency_per_chunk * 1000:.3f} ms/chunk",
        f"{avg_latency_per_signal * 1000:.3f} ms/signal",
        f"{signal_throughput:.2f} signals/sec",
        f"{avg_chunks_per_signal:.2f}"
    ]
})

class_df = pd.DataFrame({
    "Class": class_names,
    "Precision": [f"{x:.3f}" for x in class_prec],
    "Recall": [f"{x:.3f}" for x in class_rec],
    "F1": [f"{x:.3f}" for x in class_f1],
    "Support": class_support.astype(int)
})

print("=" * 70)
print(f"COMPACT SIGNAL-LEVEL TEST SUMMARY - {test_aggregation.upper()}")
print("=" * 70)

display(summary_df)

print("\nCompact per-class summary:")
display(class_df)

# -----------------------------
# Save summary table as PDF
# -----------------------------
fig, ax = plt.subplots(figsize=(8.5, 4.8))
ax.axis("off")
ax.set_title("Compact Signal-Level Test Summary", fontsize=14, pad=12)

table = ax.table(
    cellText=summary_df.values,
    colLabels=summary_df.columns,
    cellLoc="left",
    colLoc="left",
    loc="center"
)

table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1, 1.25)

plt.tight_layout()
save_pdf_figure("compact_signal_test_summary.pdf")
plt.show()

# -----------------------------
# Normalized confusion matrix only
# -----------------------------
cm = confusion_matrix(signal_targets, signal_preds_majority)

cm_norm = cm.astype(np.float32) / cm.sum(axis=1, keepdims=True)

plt.figure(figsize=(7, 5.5))
sns.heatmap(
    cm_norm,
    annot=True,
    fmt=".2f",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names
)

plt.title(f"Signal-Level Test Confusion Matrix - {test_aggregation}")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.tight_layout()

save_pdf_figure("compact_signal_test_normalized_confusion_matrix.pdf")
plt.show()


# -----------------------------
# Mean class-probability heatmap
# -----------------------------
# Rows are the true class. Columns are where the model places probability mass.
# This is a softer, more informative view than hard predictions alone.
class_probability_matrix = np.zeros((num_classes, num_classes), dtype=np.float32)

for class_idx in range(num_classes):
    mask = signal_targets == class_idx
    if np.any(mask):
        class_probability_matrix[class_idx] = signal_avg_probs[mask].mean(axis=0)

plt.figure(figsize=(7, 5.5))
sns.heatmap(
    class_probability_matrix,
    annot=True,
    fmt=".2f",
    cmap="YlOrRd",
    vmin=0,
    vmax=1,
    xticklabels=class_names,
    yticklabels=class_names
)

plt.title("Mean Model Probability by True Class")
plt.xlabel("Probability assigned to class")
plt.ylabel("True class")
plt.tight_layout()

save_pdf_figure("mean_probability_by_true_class_heatmap.pdf")
plt.show()

# -----------------------------
# Optional final attention map
# Keep this if you still want the final after-training map here
# Delete it if you already generated the after-training map earlier
# -----------------------------
plot_signal_diagnostic_dashboard(
    stage_name="After Training Test Signal",
    signal=X_test_full[0],
    label=y_test_full[0],
    compute_occlusion=False,
    aggregation=test_aggregation
)

_ = plot_confident_mistake_gallery(
    X_signals=X_test_full,
    y_signals=y_test_full,
    top_n=3,
    aggregation=test_aggregation,
    plot_examples=False
)


In [ ]:
# =========================================================
# CELL 15: FLOPs ESTIMATION BY MODULE - STRICT PURE MODEL
# =========================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# FLOPs convention:
# 1 multiply-add = 2 FLOPs.
# This is a dense software/GPU estimate.
# For pure SNN interpretation, also report sparsity-aware proxy using spike rates.

def linear_flops(batch_items, in_features, out_features):
    return batch_items * in_features * out_features * 2

def layernorm_flops(batch_items, features):
    return batch_items * features * 5

def gelu_flops(batch_items, features):
    return batch_items * features * 8

def avgpool1d_flops(batch, channels, out_len, kernel_size):
    # sum kernel values + divide
    return batch * channels * out_len * kernel_size

def spike_activation_flops(batch_items, features):
    # threshold comparison
    return batch_items * features

def slstm_cell_flops(batch, input_size, hidden_size):
    flops = 0

    # x2h and h2h
    flops += linear_flops(batch, input_size, 4 * hidden_size)
    flops += linear_flops(batch, hidden_size, 4 * hidden_size)

    # sigmoid/tanh + cell operations + membrane + threshold
    flops += batch * hidden_size * 40
    flops += batch * hidden_size

    return flops

def attention_flops(batch, timesteps, hidden_dim):
    flops = 0

    # Linear H -> H
    flops += linear_flops(batch * timesteps, hidden_dim, hidden_dim)

    # tanh
    flops += batch * timesteps * hidden_dim * 4

    # Linear H -> 1
    flops += linear_flops(batch * timesteps, hidden_dim, 1)

    # softmax
    flops += batch * timesteps * 5

    # weighted sum
    flops += batch * timesteps * hidden_dim * 2

    return flops


# ---------------------------------------------------------
# Architecture values
# ---------------------------------------------------------
B = 1
T_in = chunk_size          # 256
T_reduced = 64

C_in = input_channels      # 2
Fdim = feature_dim         # optimized: 128
H1 = hidden1               # optimized: 192
H2 = hidden2               # optimized: 128
num_cls = num_classes      # 5

multiscale_dim = C_in * 5  # 10 event features

flop_rows = []

# ---------------------------------------------------------
# Input spike encoding cost
# ---------------------------------------------------------
# This estimates the delta-sigma encoding per chunk:
# diff + residual update + pos/neg comparisons + residual correction
delta_sigma_flops = T_in * 8

flop_rows.append(["Input Delta-Sigma Encoding", delta_sigma_flops])

# ---------------------------------------------------------
# Multi-scale event-density pooling branches
# ---------------------------------------------------------
# rate_pool4: AvgPool1d over 4 samples, 256 -> 64
# rate_pool2: AvgPool1d over 2 samples, 256 -> 128 for early/late timing
rate_pool4 = avgpool1d_flops(B, C_in, T_reduced, kernel_size=4)
rate_pool2 = avgpool1d_flops(B, C_in, T_reduced * 2, kernel_size=2)

# Three threshold branches from rate4 plus early/late half-window branches from rate2
any_event = spike_activation_flops(B * T_reduced, C_in)
repeated_event = spike_activation_flops(B * T_reduced, C_in)
burst_event = spike_activation_flops(B * T_reduced, C_in)
half_window_event = spike_activation_flops(B * T_reduced * 2, C_in)

concat_cost = B * T_reduced * multiscale_dim

flop_rows.append(["AvgPool4 Spike Density", rate_pool4])
flop_rows.append(["AvgPool2 Half-Window Density", rate_pool2])
flop_rows.append(["Any-Event Spike Branch", any_event])
flop_rows.append(["Repeated-Event Spike Branch", repeated_event])
flop_rows.append(["Burst-Event Spike Branch", burst_event])
flop_rows.append(["Early/Late Event Branches", half_window_event])
flop_rows.append(["Branch Concatenation Proxy", concat_cost])

# ---------------------------------------------------------
# Input projection
# ---------------------------------------------------------
input_proj_linear = linear_flops(B * T_reduced, multiscale_dim, Fdim)
input_proj_ln = layernorm_flops(B * T_reduced, Fdim)
input_proj_spike = spike_activation_flops(B * T_reduced, Fdim)

flop_rows.append(["Input Projection Linear", input_proj_linear])
flop_rows.append(["Input Projection LayerNorm", input_proj_ln])
flop_rows.append(["Input Projection Spike Activation", input_proj_spike])

# ---------------------------------------------------------
# SLSTM recurrent block
# ---------------------------------------------------------
slstm1_per_t = slstm_cell_flops(B, Fdim, H1)
slstm2_per_t = slstm_cell_flops(B, H1, H2)

slstm1_total = slstm1_per_t * T_reduced
slstm2_total = slstm2_per_t * T_reduced

flop_rows.append(["SLSTM Layer 1 Total", slstm1_total])
flop_rows.append(["SLSTM Layer 2 Total", slstm2_total])

# ---------------------------------------------------------
# Attention + classifier
# ---------------------------------------------------------
attn_total = attention_flops(B, T_reduced, H2)

classifier_fc1 = linear_flops(B, H2, H2)
classifier_ln = layernorm_flops(B, H2)
classifier_spike = spike_activation_flops(B, H2)
classifier_fc2 = linear_flops(B, H2, num_cls)

flop_rows.append(["Attention Pooling", attn_total])
flop_rows.append(["Classifier FC1", classifier_fc1])
flop_rows.append(["Classifier LayerNorm", classifier_ln])
flop_rows.append(["Classifier Spike Activation", classifier_spike])
flop_rows.append(["Classifier FC2", classifier_fc2])

# ---------------------------------------------------------
# Build FLOPs table
# ---------------------------------------------------------
flops_df = pd.DataFrame(flop_rows, columns=["Layer / Module", "Dense FLOPs per chunk"])
total_dense_flops = flops_df["Dense FLOPs per chunk"].sum()

flops_df["Dense MFLOPs per chunk"] = flops_df["Dense FLOPs per chunk"] / 1e6
flops_df["Percent of Total"] = 100 * flops_df["Dense FLOPs per chunk"] / total_dense_flops


def assign_group(name):
    if "Encoding" in name:
        return "Input Spike Encoding"
    if "AvgPool" in name or "Branch" in name:
        return "Multi-Scale Spike Pooling"
    if "Input Projection" in name:
        return "Spike Projection"
    if "SLSTM" in name:
        return "Spiking Recurrent Block"
    if "Attention" in name:
        return "Attention Pooling"
    if "Classifier" in name:
        return "Classifier"
    return "Other"


flops_df["Group"] = flops_df["Layer / Module"].apply(assign_group)

group_flops_df = (
    flops_df
    .groupby("Group", as_index=False)["Dense FLOPs per chunk"]
    .sum()
)

group_flops_df["Dense MFLOPs per chunk"] = group_flops_df["Dense FLOPs per chunk"] / 1e6
group_flops_df["Percent of Total"] = 100 * group_flops_df["Dense FLOPs per chunk"] / total_dense_flops

# ---------------------------------------------------------
# Sparsity-aware proxy
# ---------------------------------------------------------
# Uses measured activity from evaluation if available.
# Fallbacks come from your reported final pure model stats.
input_rate = avg_input_activity if "avg_input_activity" in globals() else 0.1563
multiscale_rate = avg_multiscale_activity if "avg_multiscale_activity" in globals() else 0.1916
projected_rate = avg_projected_activity if "avg_projected_activity" in globals() else 0.4240
hidden_rate = avg_hidden_spike_rate if "avg_hidden_spike_rate" in globals() else 0.3629

def sparsity_factor_for_group(group):
    if group == "Input Spike Encoding":
        return 1.0
    if group == "Multi-Scale Spike Pooling":
        return input_rate
    if group == "Spike Projection":
        return multiscale_rate
    if group == "Spiking Recurrent Block":
        return projected_rate
    if group == "Attention Pooling":
        return hidden_rate
    if group == "Classifier":
        return hidden_rate
    return 1.0

flops_df["Sparsity Factor"] = flops_df["Group"].apply(sparsity_factor_for_group)
flops_df["Sparsity-Aware FLOPs Proxy"] = flops_df["Dense FLOPs per chunk"] * flops_df["Sparsity Factor"]
flops_df["Sparsity-Aware MFLOPs Proxy"] = flops_df["Sparsity-Aware FLOPs Proxy"] / 1e6

total_sparse_proxy_flops = flops_df["Sparsity-Aware FLOPs Proxy"].sum()

group_sparse_df = (
    flops_df
    .groupby("Group", as_index=False)[["Dense FLOPs per chunk", "Sparsity-Aware FLOPs Proxy"]]
    .sum()
)

group_sparse_df["Dense MFLOPs per chunk"] = group_sparse_df["Dense FLOPs per chunk"] / 1e6
group_sparse_df["Sparsity-Aware MFLOPs Proxy"] = group_sparse_df["Sparsity-Aware FLOPs Proxy"] / 1e6
group_sparse_df["Dense %"] = 100 * group_sparse_df["Dense FLOPs per chunk"] / total_dense_flops
group_sparse_df["Sparse Proxy %"] = 100 * group_sparse_df["Sparsity-Aware FLOPs Proxy"] / total_sparse_proxy_flops

print("=" * 80)
print("STRICT PURE MULTI-SCALE POOLED SLSTM - FLOPs ESTIMATION")
print("=" * 80)
print("Convention: 1 multiply-add = 2 FLOPs")
print("Dense FLOPs = standard software/GPU estimate")
print("Sparsity-aware FLOPs proxy = approximate event-driven compute estimate using measured spike activity")
print(f"Input chunk length:              {T_in}")
print(f"Reduced recurrent timesteps:     {T_reduced}")
print(f"Total dense FLOPs per chunk:     {total_dense_flops:,.0f}")
print(f"Total dense MFLOPs per chunk:    {total_dense_flops / 1e6:.4f}")
print(f"Sparsity-aware FLOPs proxy:      {total_sparse_proxy_flops:,.0f}")
print(f"Sparsity-aware MFLOPs proxy:     {total_sparse_proxy_flops / 1e6:.4f}")

print("\nMeasured activity rates used:")
print(f"Input spike rate:                {input_rate:.4f}")
print(f"Multiscale spike rate:           {multiscale_rate:.4f}")
print(f"Projection spike rate:           {projected_rate:.4f}")
print(f"Hidden spike rate:               {hidden_rate:.4f}")

print("\n" + "=" * 80)
print("FLOPs DISTRIBUTION BY GROUP")
print("=" * 80)
display(group_sparse_df.sort_values("Dense FLOPs per chunk", ascending=False))

print("\n" + "=" * 80)
print("FLOPs DISTRIBUTION BY LAYER / MODULE")
print("=" * 80)
display(flops_df.sort_values("Dense FLOPs per chunk", ascending=False))

# ---------------------------------------------------------
# Plot dense FLOPs by group
# ---------------------------------------------------------
plt.figure(figsize=(10, 6))
plot_df = group_sparse_df.sort_values("Dense FLOPs per chunk", ascending=True)
plt.barh(plot_df["Group"], plot_df["Dense MFLOPs per chunk"])
plt.xlabel("Dense MFLOPs per chunk")
plt.title("Strict Pure Model - Dense FLOPs Distribution by Group")
plt.tight_layout()
save_pdf_figure("dense_flops_distribution_by_group.pdf")
plt.show()

# ---------------------------------------------------------
# Plot sparsity-aware proxy by group
# ---------------------------------------------------------
plt.figure(figsize=(10, 6))
plot_df = group_sparse_df.sort_values("Sparsity-Aware FLOPs Proxy", ascending=True)
plt.barh(plot_df["Group"], plot_df["Sparsity-Aware MFLOPs Proxy"])
plt.xlabel("Sparsity-Aware MFLOPs Proxy per chunk")
plt.title("Strict Pure Model - Sparsity-Aware FLOPs Proxy by Group")
plt.tight_layout()
save_pdf_figure("sparsity_aware_flops_proxy_by_group.pdf")
plt.show()

# Store for CO2 cell
estimated_dense_flops_per_chunk = total_dense_flops
estimated_dense_mflops_per_chunk = total_dense_flops / 1e6
estimated_sparse_proxy_flops_per_chunk = total_sparse_proxy_flops
estimated_sparse_proxy_mflops_per_chunk = total_sparse_proxy_flops / 1e6


In [ ]:
# =========================================================
# CELL 16: ENERGY / CO2 EMISSIONS ESTIMATION - STRICT PURE MODEL
# =========================================================

import numpy as np
import pandas as pd

# =========================================================
# USER-ADJUSTABLE ASSUMPTIONS
# =========================================================
# Kaggle often provides T4/P100-style GPUs.
# Keep these explicit in your report as assumptions.

gpu_power_watts = 70.0
cpu_platform_overhead_watts = 30.0
pue = 1.20

# Carbon intensity estimate.
# You can change this depending on the grid/provider assumption.
carbon_intensity_kg_per_kwh = 0.45

def energy_and_co2_from_seconds(seconds, gpu_w, overhead_w, pue, carbon_intensity):
    total_power_w = (gpu_w + overhead_w) * pue
    energy_kwh = (total_power_w * seconds) / (1000.0 * 3600.0)
    co2_kg = energy_kwh * carbon_intensity
    return energy_kwh, co2_kg

# =========================================================
# Training time
# =========================================================
if "history" in globals() and len(history) > 0:
    training_time_sec = float(sum(row["time_sec"] for row in history))
    epochs_ran = len(history)
    avg_epoch_time_sec = training_time_sec / epochs_ran
else:
    training_time_sec = None
    epochs_ran = None
    avg_epoch_time_sec = None

# =========================================================
# Inference time values from evaluation cells
# =========================================================
chunk_eval_time_sec = total_eval_time if "total_eval_time" in globals() else None
signal_eval_time_sec = total_signal_eval_time if "total_signal_eval_time" in globals() else None

num_test_chunks = len(test_loader.dataset) if "test_loader" in globals() else None
num_test_signals = len(X_test_full) if "X_test_full" in globals() else None

rows = []

if training_time_sec is not None:
    train_energy_kwh, train_co2_kg = energy_and_co2_from_seconds(
        training_time_sec,
        gpu_power_watts,
        cpu_platform_overhead_watts,
        pue,
        carbon_intensity_kg_per_kwh
    )

    rows.append({
        "Stage": "Training",
        "Duration sec": training_time_sec,
        "Duration min": training_time_sec / 60.0,
        "Energy kWh": train_energy_kwh,
        "CO2 kg": train_co2_kg,
        "CO2 g": train_co2_kg * 1000.0
    })

if chunk_eval_time_sec is not None:
    chunk_energy_kwh, chunk_co2_kg = energy_and_co2_from_seconds(
        chunk_eval_time_sec,
        gpu_power_watts,
        cpu_platform_overhead_watts,
        pue,
        carbon_intensity_kg_per_kwh
    )

    rows.append({
        "Stage": "Chunk-level test inference",
        "Duration sec": chunk_eval_time_sec,
        "Duration min": chunk_eval_time_sec / 60.0,
        "Energy kWh": chunk_energy_kwh,
        "CO2 kg": chunk_co2_kg,
        "CO2 g": chunk_co2_kg * 1000.0
    })

if signal_eval_time_sec is not None:
    signal_energy_kwh, signal_co2_kg = energy_and_co2_from_seconds(
        signal_eval_time_sec,
        gpu_power_watts,
        cpu_platform_overhead_watts,
        pue,
        carbon_intensity_kg_per_kwh
    )

    rows.append({
        "Stage": "Signal-level test inference",
        "Duration sec": signal_eval_time_sec,
        "Duration min": signal_eval_time_sec / 60.0,
        "Energy kWh": signal_energy_kwh,
        "CO2 kg": signal_co2_kg,
        "CO2 g": signal_co2_kg * 1000.0
    })

co2_df = pd.DataFrame(rows)

print("=" * 80)
print("STRICT PURE MULTI-SCALE POOLED SLSTM — ENERGY / CO2 ESTIMATION")
print("=" * 80)
print("These are approximate estimates based on assumed power draw.")
print(f"GPU power assumption:        {gpu_power_watts:.1f} W")
print(f"CPU/platform overhead:       {cpu_platform_overhead_watts:.1f} W")
print(f"PUE:                         {pue:.2f}")
print(f"Carbon intensity:            {carbon_intensity_kg_per_kwh:.3f} kg CO2/kWh")

if training_time_sec is not None:
    print("\nTraining runtime:")
    print(f"Epochs ran:                  {epochs_ran}")
    print(f"Total training time:         {training_time_sec:.2f} sec")
    print(f"Total training time:         {training_time_sec / 60.0:.2f} min")
    print(f"Average epoch time:          {avg_epoch_time_sec:.2f} sec")

print("\n" + "=" * 80)
print("CO2 TABLE")
print("=" * 80)
display(co2_df)

# =========================================================
# Per-sample inference CO2
# =========================================================
print("\n" + "=" * 80)
print("PER-SAMPLE INFERENCE ENERGY / CO2")
print("=" * 80)

if chunk_eval_time_sec is not None and num_test_chunks is not None:
    chunk_energy_kwh_per_chunk = chunk_energy_kwh / num_test_chunks
    chunk_co2_g_per_chunk = (chunk_co2_kg * 1000.0) / num_test_chunks

    print(f"Chunks evaluated:            {num_test_chunks:,}")
    print(f"Energy per chunk:            {chunk_energy_kwh_per_chunk:.10f} kWh")
    print(f"CO2 per chunk:               {chunk_co2_g_per_chunk:.8f} g")

if signal_eval_time_sec is not None and num_test_signals is not None:
    signal_energy_kwh_per_signal = signal_energy_kwh / num_test_signals
    signal_co2_g_per_signal = (signal_co2_kg * 1000.0) / num_test_signals

    print(f"\nSignals evaluated:           {num_test_signals:,}")
    print(f"Energy per signal:           {signal_energy_kwh_per_signal:.10f} kWh")
    print(f"CO2 per signal:              {signal_co2_g_per_signal:.8f} g")

# =========================================================
# FLOPs summary
# =========================================================
print("\n" + "=" * 80)
print("FLOPs-BASED SUMMARY")
print("=" * 80)

if "estimated_dense_flops_per_chunk" in globals():
    print(f"Dense FLOPs per chunk:           {estimated_dense_flops_per_chunk:,.0f}")
    print(f"Dense MFLOPs per chunk:          {estimated_dense_mflops_per_chunk:.4f}")

    print(f"Sparsity-aware FLOPs proxy:      {estimated_sparse_proxy_flops_per_chunk:,.0f}")
    print(f"Sparsity-aware MFLOPs proxy:     {estimated_sparse_proxy_mflops_per_chunk:.4f}")

    if num_test_chunks is not None:
        total_dense_test_flops = estimated_dense_flops_per_chunk * num_test_chunks
        total_sparse_test_flops = estimated_sparse_proxy_flops_per_chunk * num_test_chunks

        print(f"\nTotal dense FLOPs for test chunks:       {total_dense_test_flops:,.0f}")
        print(f"Total dense GFLOPs for test set:         {total_dense_test_flops / 1e9:.4f}")

        print(f"Total sparse-proxy FLOPs for test chunks:{total_sparse_test_flops:,.0f}")
        print(f"Total sparse-proxy GFLOPs for test set:  {total_sparse_test_flops / 1e9:.4f}")
else:
    print("Run Cell 15 first to calculate FLOPs.")

# Manuscript Extensions for Model 2

These appended cells implement the four reviewer-facing tasks for the trained S-LSTM + Temporal Attention model:

1. Attention interpretability on a Bonn Set E/S seizure signal.
2. AWGN wearable-noise stress testing at 20 dB, 10 dB, and 0 dB.
3. Correct/incorrect prediction export and pairwise McNemar testing hooks.
4. Hardware deployment constraints: memory, theoretical energy, and CPU latency.

Run these cells after the original training/evaluation cells so `model`, `train_mean`, `train_std`, `X_test_full`, `y_test_full`, and `class_names` are already defined. If you get the desired Model 2 accuracy, save/download the generated checkpoint immediately.


In [ ]:
# =========================================================
# CELL 17: MANUSCRIPT TASK SETUP - MODEL 2
# MANUSCRIPT_TASKS_MODEL2_APPEND_V1
# =========================================================

import os
import time
import math
import itertools
import copy
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

try:
    from scipy.stats import binomtest
except Exception:
    binomtest = None

if os.path.exists('/kaggle/working'):
    MANUSCRIPT_DIR_MODEL2 = Path('/kaggle/working/manuscript_model2')
else:
    MANUSCRIPT_DIR_MODEL2 = Path.cwd() / 'manuscript_model2'

MANUSCRIPT_DIR_MODEL2.mkdir(parents=True, exist_ok=True)

BONN_SAMPLING_RATE_HZ = 4097 / 23.6
SNR_LEVELS_DB = [20, 10, 0]
MODEL2_NAME = 'Model 2 S-LSTM + Temporal Attention'

# Ensure the manuscript cells use the active best-validation checkpoint if it exists.
if 'use_best_checkpoint_for_test' in globals() and use_best_checkpoint_for_test and 'best_state' in globals() and best_state is not None:
    model.load_state_dict(best_state)
    active_checkpoint_name = f'best validation checkpoint from epoch {best_epoch}'
elif 'active_checkpoint_name' not in globals():
    active_checkpoint_name = 'current model weights'

model.to(device)
model.eval()

model2_checkpoint_path = MANUSCRIPT_DIR_MODEL2 / 'model2_current_for_manuscript.pth'
torch.save({
    'model_state_dict': copy.deepcopy(model.state_dict()),
    'best_state': copy.deepcopy(best_state) if 'best_state' in globals() and best_state is not None else None,
    'best_epoch': best_epoch if 'best_epoch' in globals() else None,
    'best_val_f1': best_val_f1 if 'best_val_f1' in globals() else None,
    'train_mean': float(train_mean),
    'train_std': float(train_std),
    'class_names': class_names,
    'chunk_size': int(chunk_size),
    'stride': int(stride),
    'input_spike_threshold': float(input_spike_threshold),
    'feature_dim': int(feature_dim),
    'hidden1': int(hidden1),
    'hidden2': int(hidden2),
    'projection_spike_threshold': float(projection_spike_threshold),
    'hidden_spike_threshold': float(hidden_spike_threshold),
}, model2_checkpoint_path)

print('=' * 80)
print('MODEL 2 MANUSCRIPT TASK SETUP')
print('=' * 80)
print(f'Output directory: {MANUSCRIPT_DIR_MODEL2}')
print(f'Active checkpoint: {active_checkpoint_name}')
print(f'Checkpoint saved: {model2_checkpoint_path}')
print(f'Bonn sampling rate: {BONN_SAMPLING_RATE_HZ:.4f} Hz')
print(f'SNR stress-test levels: {SNR_LEVELS_DB}')


def manuscript2_savefig(filename, dpi=600):
    path = MANUSCRIPT_DIR_MODEL2 / filename
    plt.savefig(path, dpi=dpi, bbox_inches='tight')
    print(f'Saved: {path}')
    return path


def model2_label_name(label_idx):
    return class_names[int(label_idx)]


def _model2_split_pool(name):
    X_name = f'X_{name}_full'
    y_name = f'y_{name}_full'
    if X_name in globals() and y_name in globals():
        return globals()[X_name], globals()[y_name]
    return None, None


def select_model2_bonn_set_e_signal(preferred_splits=('test', 'val', 'train'), seizure_index=0):
    s_idx = class_names.index('S')
    for split_name in preferred_splits:
        X_split, y_split = _model2_split_pool(split_name)
        if X_split is None:
            continue
        matches = np.where(np.asarray(y_split) == s_idx)[0]
        if len(matches) > 0:
            pos = int(matches[seizure_index % len(matches)])
            return X_split[pos], int(y_split[pos]), split_name, pos

    if 'X_full' in globals() and 'y_full' in globals():
        matches = np.where(np.asarray(y_full) == s_idx)[0]
        if len(matches) > 0:
            pos = int(matches[seizure_index % len(matches)])
            return X_full[pos], int(y_full[pos]), 'full', pos

    raise ValueError('No Bonn Set E/S seizure signal found in available arrays.')


def model2_chunk_starts_for_signal(signal_length, chunk_size=chunk_size, stride=stride):
    return np.array(list(range(0, signal_length - chunk_size + 1, stride)), dtype=np.int64)


def model2_normalize_to_unit(x):
    x = np.asarray(x, dtype=np.float64)
    lo = np.nanmin(x)
    hi = np.nanmax(x)
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        return np.zeros_like(x, dtype=np.float64)
    return (x - lo) / (hi - lo)


In [ ]:
# =========================================================
# TASK 1: INTERPRETABILITY VISUALIZATION - MODEL 2 ATTENTION MAPPING
# =========================================================

@torch.no_grad()
def create_model2_attention_interpretability_figure(
    signal=None,
    label=None,
    seizure_index=0,
    preferred_splits=('test', 'val', 'train'),
    filename_prefix='task1_model2_attention_interpretability',
):
    """
    Creates a vertically stacked manuscript figure:
    1. Raw continuous EEG waveform.
    2. Input delta-sigma spike density.
    3. Temporal attention weights projected back to EEG time.
    """
    model.eval()

    if signal is None or label is None:
        signal, label, source_split, source_pos = select_model2_bonn_set_e_signal(
            preferred_splits=preferred_splits,
            seizure_index=seizure_index,
        )
    else:
        source_split, source_pos = 'manual', seizure_index

    signal = np.asarray(signal, dtype=np.float32)
    chunks_tensor, chunks_spike = signal_to_spike_tensor(signal)

    logits, stats = model(
        chunks_tensor,
        return_spike_stats=True,
        return_attention=True,
    )

    probs = torch.softmax(logits, dim=1).mean(dim=0).detach().cpu().numpy()
    pred_idx = int(np.argmax(probs))

    attention = stats['attention_weights'].detach().cpu().numpy()  # [chunks, reduced_time]
    spike_any = (chunks_spike.sum(axis=2) > 0).astype(np.float32)  # [chunks, 256]

    n_samples = len(signal)
    starts = model2_chunk_starts_for_signal(n_samples, chunk_size=chunk_size, stride=stride)

    # Input spike density is already at raw chunk resolution.
    spike_trace = np.zeros(n_samples, dtype=np.float64)
    spike_counts = np.zeros(n_samples, dtype=np.float64)
    for chunk_idx, chunk_start in enumerate(starts):
        if chunk_idx >= spike_any.shape[0]:
            break
        seg_end = min(chunk_start + chunk_size, n_samples)
        usable = seg_end - chunk_start
        spike_trace[chunk_start:seg_end] += spike_any[chunk_idx, :usable]
        spike_counts[chunk_start:seg_end] += 1.0
    valid_spike = spike_counts > 0
    spike_trace[valid_spike] /= spike_counts[valid_spike]

    # Attention has 64 reduced steps, each representing 4 input samples.
    reduced_steps = attention.shape[1]
    samples_per_reduced_step = max(1, int(round(chunk_size / reduced_steps)))
    attention_trace = np.zeros(n_samples, dtype=np.float64)
    attention_counts = np.zeros(n_samples, dtype=np.float64)

    for chunk_idx, chunk_start in enumerate(starts):
        if chunk_idx >= attention.shape[0]:
            break
        for step in range(reduced_steps):
            seg_start = int(chunk_start + step * samples_per_reduced_step)
            seg_end = int(min(chunk_start + (step + 1) * samples_per_reduced_step, n_samples))
            if seg_start >= n_samples or seg_end <= seg_start:
                continue
            attention_trace[seg_start:seg_end] += attention[chunk_idx, step]
            attention_counts[seg_start:seg_end] += 1.0

    valid_attn = attention_counts > 0
    attention_trace[valid_attn] /= attention_counts[valid_attn]

    spike_unit = model2_normalize_to_unit(spike_trace)
    attention_unit = model2_normalize_to_unit(attention_trace)
    time_axis = np.arange(n_samples) / BONN_SAMPLING_RATE_HZ

    true_name = model2_label_name(label)
    pred_name = model2_label_name(pred_idx)

    fig, axes = plt.subplots(
        3,
        1,
        figsize=(13, 8.2),
        sharex=True,
        gridspec_kw={'height_ratios': [2.0, 1.15, 1.15]},
    )

    axes[0].plot(time_axis, signal, color='#111827', linewidth=0.8)
    axes[0].set_ylabel('Raw EEG')
    axes[0].set_title(
        f'Model 2 Attention Alignment on Bonn Set E/S Seizure '
        f'(source={source_split}[{source_pos}], true={true_name}, pred={pred_name})'
    )
    axes[0].grid(True, alpha=0.18)

    spike_nonzero = spike_unit > 0
    axes[1].vlines(
        time_axis[spike_nonzero],
        0,
        spike_unit[spike_nonzero],
        color='#0f766e',
        linewidth=0.35,
    )
    axes[1].set_ylabel('Input spikes\nnormalized')
    axes[1].set_ylim(0, 1.02)
    axes[1].grid(True, alpha=0.18)

    axes[2].plot(time_axis, attention_unit, color='#be123c', linewidth=1.35)
    axes[2].fill_between(time_axis, 0, attention_unit, color='#fecdd3', alpha=0.55)
    axes[2].set_ylabel('Attention\nnormalized')
    axes[2].set_xlabel('Time (s)')
    axes[2].set_ylim(0, 1.02)
    axes[2].grid(True, alpha=0.18)

    fig.tight_layout()
    png_path = manuscript2_savefig(f'{filename_prefix}.png')
    pdf_path = MANUSCRIPT_DIR_MODEL2 / f'{filename_prefix}.pdf'
    fig.savefig(pdf_path, bbox_inches='tight')
    print(f'Saved: {pdf_path}')
    plt.show()

    trace_df = pd.DataFrame({
        'time_sec': time_axis,
        'raw_eeg': signal,
        'input_spike_density': spike_trace,
        'input_spike_density_normalized': spike_unit,
        'attention_weight': attention_trace,
        'attention_weight_normalized': attention_unit,
    })
    trace_path = MANUSCRIPT_DIR_MODEL2 / f'{filename_prefix}_traces.csv'
    trace_df.to_csv(trace_path, index=False)
    print(f'Saved traces: {trace_path}')

    return {
        'figure_png': png_path,
        'figure_pdf': pdf_path,
        'trace_csv': trace_path,
        'true_label': true_name,
        'pred_label': pred_name,
        'source_split': source_split,
        'source_index': source_pos,
        'probabilities': probs,
    }


task1_model2_attention_result = create_model2_attention_interpretability_figure(seizure_index=0)
task1_model2_attention_result


In [ ]:
# =========================================================
# TASK 2: WEARABLE NOISE-ROBUSTNESS STRESS TEST - MODEL 2 AWGN
# =========================================================


def inject_awgn_model2(signals, snr_db, seed=SEED):
    """Inject additive white Gaussian noise into raw full EEG signals at the requested SNR in dB."""
    rng = np.random.default_rng(seed)
    signals = np.asarray(signals, dtype=np.float32)
    noisy = np.empty_like(signals, dtype=np.float32)

    for i, signal in enumerate(signals):
        signal_power = float(np.mean(signal ** 2))
        noise_power = signal_power / (10.0 ** (float(snr_db) / 10.0))
        noise = rng.normal(0.0, math.sqrt(noise_power), size=signal.shape).astype(np.float32)
        noisy[i] = signal + noise

    return noisy


@torch.no_grad()
def run_model2_awgn_stress_test(snrs_db=SNR_LEVELS_DB, aggregation='avg_probs'):
    model.eval()
    rows = []

    for snr_db in snrs_db:
        X_test_noisy = inject_awgn_model2(X_test_full, snr_db=snr_db, seed=SEED + int(100 * float(snr_db)))
        metrics = evaluate_signal_level(
            model=model,
            X_signals=X_test_noisy,
            y_signals=y_test_full,
            criterion=criterion if 'criterion' in globals() else None,
            aggregation=aggregation,
        )

        rows.append({
            'Model': MODEL2_NAME,
            'SNR_dB': float(snr_db),
            'Signal_Accuracy': float(metrics['acc']),
            'Signal_Macro_F1': float(metrics['f1']),
            'Macro_Precision': float(metrics['precision']),
            'Macro_Recall': float(metrics['recall']),
            'Input_Spike_Rate': float(metrics['input_activity']),
            'Hidden_Spike_Rate': float(metrics['hidden_spike_rate']),
        })

        print(
            f'SNR {snr_db:>5} dB | '
            f'Acc={metrics["acc"]:.4f} | F1={metrics["f1"]:.4f} | '
            f'Input spikes={metrics["input_activity"]:.4f} | Hidden spikes={metrics["hidden_spike_rate"]:.4f}'
        )

    df = pd.DataFrame(rows).sort_values('SNR_dB', ascending=False)
    csv_path = MANUSCRIPT_DIR_MODEL2 / 'task2_model2_awgn_robustness.csv'
    df.to_csv(csv_path, index=False)
    print(f'Saved: {csv_path}')

    plot_df = df.sort_values('SNR_dB')
    plt.figure(figsize=(7.2, 4.6))
    plt.plot(plot_df['SNR_dB'], plot_df['Signal_Accuracy'], marker='o', linewidth=2.0, label=MODEL2_NAME)
    plt.xlabel('SNR (dB)')
    plt.ylabel('Classification Accuracy')
    plt.title('Model 2 Noise Robustness Under AWGN')
    plt.ylim(0, 1.02)
    plt.grid(True, alpha=0.25)
    plt.legend(frameon=False)
    plt.tight_layout()
    manuscript2_savefig('task2_model2_accuracy_vs_snr.png')
    pdf_path = MANUSCRIPT_DIR_MODEL2 / 'task2_model2_accuracy_vs_snr.pdf'
    plt.savefig(pdf_path, bbox_inches='tight')
    print(f'Saved: {pdf_path}')
    plt.show()

    return df


task2_model2_noise_df = run_model2_awgn_stress_test()
task2_model2_noise_df


In [ ]:
# =========================================================
# TASK 3: STATISTICAL SIGNIFICANCE - MODEL 2 PREDICTION EXPORT + MCNEMAR HOOKS
# =========================================================

@torch.no_grad()
def export_model2_signal_predictions(aggregation='avg_probs'):
    model.eval()

    rows = []
    signal_targets = []
    signal_preds = []
    signal_avg_probs = []
    input_acts = []
    hidden_spks = []
    signal_times = []
    chunks_per_signal = []

    # Warm-up for fairer latency metadata.
    warm_tensor, _ = signal_to_spike_tensor(X_test_full[0])
    _ = model(warm_tensor)
    if torch.cuda.is_available():
        torch.cuda.synchronize()

    start_total = time.time()

    for i, (signal, label) in enumerate(zip(X_test_full, y_test_full)):
        chunks_tensor, _ = signal_to_spike_tensor(signal)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        start_signal = time.time()
        logits, stats = model(chunks_tensor, return_spike_stats=True)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        signal_times.append(time.time() - start_signal)

        probs = torch.softmax(logits, dim=1)
        avg_probs = probs.mean(dim=0).detach().cpu().numpy()
        chunk_preds = torch.argmax(probs, dim=1).detach().cpu().numpy()

        if aggregation == 'majority':
            pred = int(np.bincount(chunk_preds, minlength=num_classes).argmax())
        elif aggregation == 'avg_logits':
            pred = int(torch.argmax(logits.mean(dim=0)).item())
        elif aggregation == 'avg_probs':
            pred = int(np.argmax(avg_probs))
        else:
            raise ValueError('aggregation must be one of: majority, avg_logits, avg_probs')

        signal_targets.append(int(label))
        signal_preds.append(pred)
        signal_avg_probs.append(avg_probs)
        input_acts.append(float(stats['input_activity'].item()))
        hidden_spks.append(float(stats['hidden_spike_rate'].item()))
        chunks_per_signal.append(int(chunks_tensor.size(0)))

        row = {
            'Model': MODEL2_NAME,
            'Signal_Index': i,
            'True_Label_Index': int(label),
            'True_Label': model2_label_name(label),
            'Pred_Label_Index': int(pred),
            'Pred_Label': model2_label_name(pred),
            'Correct': int(int(label) == pred),
        }
        for class_idx, class_name in enumerate(class_names):
            row[f'Prob_{class_name}'] = float(avg_probs[class_idx])
        rows.append(row)

    total_time = time.time() - start_total
    targets = np.asarray(signal_targets, dtype=np.int64)
    preds = np.asarray(signal_preds, dtype=np.int64)
    avg_probs_all = np.vstack(signal_avg_probs)

    prec, rec, f1, _ = precision_recall_fscore_support(
        targets,
        preds,
        average='macro',
        zero_division=0,
    )
    acc = accuracy_score(targets, preds)

    metrics = {
        'targets': targets,
        'preds': preds,
        'avg_probs': avg_probs_all,
        'correct': (targets == preds).astype(np.int64),
        'acc': float(acc),
        'precision': float(prec),
        'recall': float(rec),
        'f1': float(f1),
        'input_activity': float(np.mean(input_acts)),
        'hidden_spike_rate': float(np.mean(hidden_spks)),
        'avg_latency_per_signal_sec': float(np.mean(signal_times)),
        'avg_chunks_per_signal': float(np.mean(chunks_per_signal)),
        'avg_latency_per_chunk_sec': float(np.mean(signal_times) / np.mean(chunks_per_signal)),
        'total_time_sec': float(total_time),
        'total_chunks': int(np.sum(chunks_per_signal)),
    }

    df = pd.DataFrame(rows)
    csv_path = MANUSCRIPT_DIR_MODEL2 / 'task3_model2_signal_predictions_correctness.csv'
    df.to_csv(csv_path, index=False)
    print(f'Saved: {csv_path}')
    print(f'Model 2 clean signal accuracy: {metrics["acc"]:.4f}')
    print(f'Model 2 clean signal macro-F1: {metrics["f1"]:.4f}')

    checkpoint_path = MANUSCRIPT_DIR_MODEL2 / f'model2_acc_{metrics["acc"]:.4f}.pth'
    torch.save({
        'model_state_dict': copy.deepcopy(model.state_dict()),
        'best_state': copy.deepcopy(best_state) if 'best_state' in globals() and best_state is not None else None,
        'best_epoch': best_epoch if 'best_epoch' in globals() else None,
        'best_val_f1': best_val_f1 if 'best_val_f1' in globals() else None,
        'train_mean': float(train_mean),
        'train_std': float(train_std),
        'test_acc': float(metrics['acc']),
        'test_f1': float(metrics['f1']),
        'class_names': class_names,
        'input_spike_threshold': float(input_spike_threshold),
    }, checkpoint_path)
    print(f'Saved locked Model 2 checkpoint: {checkpoint_path}')

    return df, metrics


def mcnemar_exact_or_corrected_model2(correct_a, correct_b):
    a = np.asarray(correct_a).astype(bool)
    b = np.asarray(correct_b).astype(bool)
    a_correct_b_wrong = int(np.sum(a & ~b))
    a_wrong_b_correct = int(np.sum(~a & b))
    discordant = a_correct_b_wrong + a_wrong_b_correct

    if discordant == 0:
        return {
            'A_Correct_B_Wrong': a_correct_b_wrong,
            'A_Wrong_B_Correct': a_wrong_b_correct,
            'Discordant_Pairs': discordant,
            'P_Value': 1.0,
            'Method': 'no_discordant_pairs',
        }

    if binomtest is not None:
        p_value = float(
            binomtest(
                min(a_correct_b_wrong, a_wrong_b_correct),
                n=discordant,
                p=0.5,
                alternative='two-sided',
            ).pvalue
        )
        method = 'exact_binomial_mcnemar'
    else:
        statistic = (abs(a_correct_b_wrong - a_wrong_b_correct) - 1.0) ** 2 / discordant
        p_value = float(math.erfc(math.sqrt(statistic / 2.0)))
        method = 'chi_square_continuity_mcnemar'

    return {
        'A_Correct_B_Wrong': a_correct_b_wrong,
        'A_Wrong_B_Correct': a_wrong_b_correct,
        'Discordant_Pairs': discordant,
        'P_Value': p_value,
        'Method': method,
    }


def run_pairwise_mcnemar_from_prediction_csvs_model2(named_csv_paths):
    """
    Compare any number of model prediction CSVs.

    Example after all models are ready:
        run_pairwise_mcnemar_from_prediction_csvs_model2({
            'Model 1 Conventional LSTM': '/path/to/model1_predictions.csv',
            'Model 2 S-LSTM Attention': MANUSCRIPT_DIR_MODEL2 / 'task3_model2_signal_predictions_correctness.csv',
            'Model 3 Conv-SLSTM Attention': '/path/to/task3_model3_signal_predictions_correctness.csv',
        })
    """
    tables = {}
    for name, csv_path in named_csv_paths.items():
        df = pd.read_csv(csv_path).sort_values('Signal_Index').reset_index(drop=True)
        tables[name] = df

    rows = []
    for name_a, name_b in itertools.combinations(tables.keys(), 2):
        a = tables[name_a]
        b = tables[name_b]
        merged = a[['Signal_Index', 'True_Label_Index', 'Correct']].merge(
            b[['Signal_Index', 'True_Label_Index', 'Correct']],
            on=['Signal_Index', 'True_Label_Index'],
            suffixes=('_A', '_B'),
        )
        result = mcnemar_exact_or_corrected_model2(merged['Correct_A'], merged['Correct_B'])
        result.update({
            'Model_A': name_a,
            'Model_B': name_b,
            'Aligned_Test_Signals': len(merged),
            'Significant_p_lt_0_05': bool(result['P_Value'] < 0.05),
        })
        rows.append(result)

    out_df = pd.DataFrame(rows)
    csv_path = MANUSCRIPT_DIR_MODEL2 / 'task3_pairwise_mcnemar_results.csv'
    out_df.to_csv(csv_path, index=False)
    print(f'Saved: {csv_path}')
    return out_df


task3_model2_predictions_df, task3_model2_clean_metrics = export_model2_signal_predictions()
task3_model2_predictions_df.head()


In [ ]:
# =========================================================
# TASK 4: COMPREHENSIVE HARDWARE PROFILING - MODEL 2
# =========================================================


def model2_parameter_memory_table(model):
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    return pd.DataFrame([{
        'Model': MODEL2_NAME,
        'Total_Parameters': int(total_params),
        'Trainable_Parameters': int(trainable_params),
        'Float32_Memory_KB': total_params * 4 / 1024,
        'Float32_Memory_MB': total_params * 4 / (1024 ** 2),
        'Int8_Quantized_Memory_KB': total_params / 1024,
        'One_Bit_Theoretical_Memory_KB': total_params / 8 / 1024,
    }])


def model2_linear_macs(batch_items, in_features, out_features):
    return int(batch_items * in_features * out_features)


def estimate_model2_mac_ac_counts(input_activity=None, hidden_spike_rate=None):
    """
    Approximate dense MAC/AC counts and spike-aware MAC/AC counts per 256-sample chunk.
    Input delta-sigma and pooling are treated as AC/compare operations; S-LSTM recurrent
    projections are event-rate adjusted under the spike-aware estimate.
    """
    B = 1
    T_in = int(chunk_size)
    T_reduced = T_in // 4
    C_in = int(input_channels)
    multiscale_dim = C_in * 5
    Fdim = int(feature_dim)
    H1 = int(hidden1)
    H2 = int(hidden2)
    num_cls = int(num_classes)

    if input_activity is None:
        input_activity = float(task3_model2_clean_metrics.get('input_activity', 0.10)) if 'task3_model2_clean_metrics' in globals() else 0.10
    if hidden_spike_rate is None:
        hidden_spike_rate = float(task3_model2_clean_metrics.get('hidden_spike_rate', 0.10)) if 'task3_model2_clean_metrics' in globals() else 0.10

    dense_macs = 0
    dense_acs = 0
    spike_aware_macs = 0
    spike_aware_acs = 0

    def add_dense(macs=0, acs=0):
        nonlocal dense_macs, dense_acs, spike_aware_macs, spike_aware_acs
        dense_macs += int(macs)
        dense_acs += int(acs)
        spike_aware_macs += int(macs)
        spike_aware_acs += int(acs)

    # Delta-sigma input encoding and rate pooling / spike comparisons.
    add_dense(0, T_in * C_in * 8)
    add_dense(0, B * C_in * T_reduced * 4)
    add_dense(0, B * C_in * (T_reduced * 2) * 2)
    add_dense(0, B * T_reduced * C_in * 5)

    # Input projection: under event-driven interpretation, sparse input spikes reduce MAC-like work.
    input_proj_macs = model2_linear_macs(B * T_reduced, multiscale_dim, Fdim)
    dense_macs += input_proj_macs
    dense_acs += B * T_reduced * Fdim * 6
    spike_aware_acs += int(input_proj_macs * np.clip(input_activity, 0.0, 1.0))
    spike_aware_acs += B * T_reduced * Fdim * 6

    # S-LSTM projections.
    slstm1_x = model2_linear_macs(B * T_reduced, Fdim, 4 * H1)
    slstm1_h = model2_linear_macs(B * T_reduced, H1, 4 * H1)
    slstm2_x = model2_linear_macs(B * T_reduced, H1, 4 * H2)
    slstm2_h = model2_linear_macs(B * T_reduced, H2, 4 * H2)
    slstm_elementwise = B * T_reduced * (H1 + H2) * 41

    dense_macs += slstm1_x + slstm1_h + slstm2_x + slstm2_h
    dense_acs += slstm_elementwise
    spike_aware_macs += slstm1_x + slstm2_x
    spike_aware_acs += int((slstm1_h + slstm2_h) * np.clip(hidden_spike_rate, 0.0, 1.0))
    spike_aware_acs += slstm_elementwise

    # Attention and classifier.
    add_dense(model2_linear_macs(B * T_reduced, H2, H2), B * T_reduced * (H2 * 4 + 5))
    add_dense(model2_linear_macs(B * T_reduced, H2, 1), 0)
    add_dense(model2_linear_macs(B, H2, H2), B * H2 * 6)
    add_dense(model2_linear_macs(B, H2, num_cls), 0)

    return {
        'Dense_MACs_Per_Chunk': float(dense_macs),
        'Dense_ACs_Per_Chunk': float(dense_acs),
        'Spike_Aware_MACs_Per_Chunk': float(spike_aware_macs),
        'Spike_Aware_ACs_Per_Chunk': float(spike_aware_acs),
        'Input_Spike_Rate_Used': float(input_activity),
        'Hidden_Spike_Rate_Used': float(hidden_spike_rate),
    }


@torch.no_grad()
def measure_model2_cpu_latency_per_chunk_ms(repeats=100, warmup=20):
    profile_model = copy.deepcopy(model).to('cpu')
    profile_model.eval()
    x = torch.zeros(1, chunk_size, input_channels, dtype=torch.float32)

    for _ in range(warmup):
        _ = profile_model(x)

    start = time.perf_counter()
    for _ in range(repeats):
        _ = profile_model(x)
    elapsed = time.perf_counter() - start

    del profile_model
    return (elapsed / repeats) * 1000.0


def run_model2_hardware_profile(mac_energy_pj=4.6, ac_energy_pj=0.9, repeats=100, warmup=20):
    mem_df = model2_parameter_memory_table(model)

    if 'task3_model2_clean_metrics' not in globals():
        _, clean_metrics = export_model2_signal_predictions()
    else:
        clean_metrics = task3_model2_clean_metrics

    ops = estimate_model2_mac_ac_counts(
        input_activity=clean_metrics.get('input_activity', None),
        hidden_spike_rate=clean_metrics.get('hidden_spike_rate', None),
    )

    cpu_latency_ms = measure_model2_cpu_latency_per_chunk_ms(repeats=repeats, warmup=warmup)
    avg_chunks_per_signal = float(clean_metrics.get('avg_chunks_per_signal', np.nan))

    dense_energy_j = (ops['Dense_MACs_Per_Chunk'] * mac_energy_pj + ops['Dense_ACs_Per_Chunk'] * ac_energy_pj) * 1e-12
    spike_energy_j = (ops['Spike_Aware_MACs_Per_Chunk'] * mac_energy_pj + ops['Spike_Aware_ACs_Per_Chunk'] * ac_energy_pj) * 1e-12

    hw_row = mem_df.iloc[0].to_dict()
    hw_row.update(ops)
    hw_row.update({
        'MAC_Energy_pJ': float(mac_energy_pj),
        'AC_Energy_pJ': float(ac_energy_pj),
        'Dense_Energy_J_Per_Chunk': float(dense_energy_j),
        'Spike_Aware_Energy_J_Per_Chunk': float(spike_energy_j),
        'CPU_Latency_ms_Per_Chunk': float(cpu_latency_ms),
        'Avg_Chunks_Per_Signal': avg_chunks_per_signal,
        'CPU_Latency_ms_Per_Signal_Est': float(cpu_latency_ms * avg_chunks_per_signal),
        'Spike_Aware_Energy_J_Per_Signal_Est': float(spike_energy_j * avg_chunks_per_signal),
    })

    hw_df = pd.DataFrame([hw_row])
    csv_path = MANUSCRIPT_DIR_MODEL2 / 'task4_model2_hardware_deployment_constraints.csv'
    hw_df.to_csv(csv_path, index=False)
    print(f'Saved: {csv_path}')

    display_cols = [
        'Model',
        'Total_Parameters',
        'Float32_Memory_KB',
        'Int8_Quantized_Memory_KB',
        'One_Bit_Theoretical_Memory_KB',
        'Spike_Aware_Energy_J_Per_Chunk',
        'CPU_Latency_ms_Per_Chunk',
        'CPU_Latency_ms_Per_Signal_Est',
    ]

    fig, ax = plt.subplots(figsize=(13, 2.8))
    ax.axis('off')
    ax.set_title('Hardware Deployment Constraints - Model 2', fontsize=13, pad=10)

    table_df = hw_df[display_cols].copy()
    for col in table_df.columns:
        if col != 'Model':
            table_df[col] = table_df[col].map(lambda x: f'{x:.6g}')

    table = ax.table(
        cellText=table_df.values,
        colLabels=table_df.columns,
        cellLoc='center',
        loc='center',
    )
    table.auto_set_font_size(False)
    table.set_fontsize(7.3)
    table.scale(1, 1.45)
    plt.tight_layout()
    manuscript2_savefig('task4_model2_hardware_deployment_constraints.png')
    pdf_path = MANUSCRIPT_DIR_MODEL2 / 'task4_model2_hardware_deployment_constraints.pdf'
    plt.savefig(pdf_path, bbox_inches='tight')
    print(f'Saved: {pdf_path}')
    plt.show()

    return hw_df


task4_model2_hardware_df = run_model2_hardware_profile(repeats=100, warmup=20)
task4_model2_hardware_df
